<div style="color: #0B39C9; font-size: 24px; font-weight: 700;">

DỰ ÁN: XÂY DỰNG MÔ HÌNH PHÂN LOẠI VÀ DỰ BÁO RỦI RO KHÁCH HÀNG VAY VỐN

</div>

# <span style="color: #E60000;">Notebook 06. Machine Learning</span>

*Huấn luyện, đánh giá và lựa chọn mô hình phân loại rủi ro tín dụng từ bộ đặc trưng `application_features` để bàn giao sang Notebook 07*

---

**Input:** Bảng `application_features` trong PostgreSQL (kết quả của Notebook 05), gồm dữ liệu đã làm sạch và các đặc trưng được lựa chọn cho mô hình.

**Output:** Mô hình được lựa chọn, các artifact tiền xử lý cần thiết và metadata gồm danh sách feature, chỉ số đánh giá cùng ngưỡng dự đoán để bàn giao sang Notebook 07.

**Pipeline:** Business Understanding → Data Understanding → Database Organization → Data Cleaning → EDA & Visualization → Feature Engineering → **Machine Learning** → Prediction Demo


## I. Giới thiệu

### 1. Mục tiêu của Notebook 06

Huấn luyện và so sánh Logistic Regression, Random Forest và HistGradientBoosting để chọn mô hình dự đoán rủi ro tín dụng phù hợp nhất cho bài toán.

### 2. Vai trò của Machine Learning trong dự án


Mô hình biến các đặc trưng đã chuẩn bị thành xác suất rủi ro, từ đó hỗ trợ đánh giá hồ sơ vay một cách nhất quán thay vì phụ thuộc cảm tính của từng nhân viên tín dụng.

### 3. Mạch liên kết: Notebook 05 → Notebook 06 → Notebook 07

Notebook 05 bàn giao bảng `application_features`; Notebook 06 chọn mô hình, chốt ngưỡng quyết định và lưu trọn bộ pipeline tiền xử lý để Notebook 07 thực hiện dự đoán trên hồ sơ mới.

## II. Đọc dữ liệu

### 1. Kết nối PostgreSQL và đọc bảng `application_features`

In [1]:
# Toan bo thu vien dung trong Notebook 06 duoc gom tai cell nay.
import json
import os
import time
from datetime import datetime
from pathlib import Path

import joblib
from sklearn.base import clone
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sqlalchemy import URL, create_engine

# Xac dinh thu muc goc repo de moi artifact luon duoc luu vao <goc>/models,
# bat ke notebook duoc chay tu thu muc nao (VSCode, Jupyter hay nbconvert).
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "AGENTS.md").exists():
    raise FileNotFoundError("Khong xac dinh duoc thu muc goc repo (khong thay AGENTS.md).")

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
print(f"Thu muc goc du an: {PROJECT_ROOT}")
print(f"Thu muc luu mo hinh: {MODELS_DIR}")

Thu muc goc du an: D:\du an 1
Thu muc luu mo hinh: D:\du an 1\models


In [2]:
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
if not env_path.exists():
    raise FileNotFoundError("Không tìm thấy file .env chứa cấu hình PostgreSQL.")

load_dotenv(env_path)
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME", "credit_risk_db"),
)
engine = create_engine(db_url)

df = pd.read_sql("SELECT * FROM public.application_features", engine)
assert {"sk_id_curr", "target"}.issubset(df.columns), (
    "Bảng application_features phải có sk_id_curr và target."
)
print(f"Đã đọc application_features: {df.shape[0]:,} dòng × {df.shape[1]:,} cột")
display(df.head())


Đã đọc application_features: 305,181 dòng × 157 cột


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,bureau_recency_days,previous_credit_to_current,previous_recency_days,installments_payment_ratio,has_installments_late,has_pos_cash_dpd,credit_card_utilization,age_income_interaction,late_debt_interaction,ext_ltv_interaction
0,101945,0,Cash loans,M,Y,N,0,90000.0,1078200.0,34911.0,...,358.0,0.093907,1355.0,1.000000,0,0,NaN,45-54 | Rất thấp,Không ghi nhận trễ | Không còn dư nợ,0.514418
1,101946,0,Cash loans,F,N,Y,0,225000.0,814041.0,28971.0,...,16.0,0.255923,155.0,0.973430,1,1,NaN,25-34 | Cao,Từng trễ | Cao,0.624708
2,101947,0,Cash loans,F,N,Y,0,360000.0,1845000.0,65088.0,...,414.0,0.016639,2396.0,1.000000,0,0,NaN,45-54 | Rất cao,Không ghi nhận trễ | Không còn dư nợ,0.339894
3,101948,0,Cash loans,F,Y,Y,0,157500.0,369000.0,26869.5,...,510.0,1.241983,130.0,0.935223,1,0,0.732157,45-54 | Trung bình,Từng trễ | Thấp,0.573530
4,101949,0,Cash loans,M,N,Y,0,180000.0,473760.0,53712.0,...,393.0,0.357423,71.0,0.990062,1,0,0.314303,35-44 | Cao,NaN,0.461062


**Nhận xét:** Đọc được 305.181 dòng × 157 cột từ `application_features`. Đây là đầu vào duy nhất của NB06 — không đọc lại CSV thô, đảm bảo mọi bước làm sạch và tạo đặc trưng ở NB03/NB05 đều được kế thừa.

### 2. Kiểm tra kích thước và kiểu dữ liệu


In [3]:
input_summary = pd.DataFrame({
    "Chỉ tiêu": ["Số dòng", "Số cột", "Số cột số", "Số cột phân loại", "Tổng giá trị thiếu"],
    "Kết quả": [
        len(df),
        df.shape[1],
        df.select_dtypes(include="number").shape[1],
        df.select_dtypes(exclude="number").shape[1],
        int(df.isna().sum().sum()),
    ],
})

display(input_summary)
display(
    df.dtypes.value_counts()
    .rename_axis("Kiểu dữ liệu")
    .reset_index(name="Số cột")
)


,Chỉ tiêu,Kết quả
0,Số dòng,305181
1,Số cột,157
2,Số cột số,138
3,Số cột phân loại,19
4,Tổng giá trị thiếu,2725936


,Kiểu dữ liệu,Số cột
0,float64,89
1,int64,49
2,str,19


**Nhận xét:**
- 138 cột số, 19 cột phân loại, tổng cộng 2.725.936 giá trị thiếu (~5,7% số ô).
- Lượng khuyết thiếu này chủ yếu đến từ các bảng phụ (`bureau`, `previous_application`) mà không phải khách hàng nào cũng có lịch sử.
- **Đề xuất:** bắt buộc điền khuyết ở Mục III.4 trước khi huấn luyện, vì Logistic Regression và Random Forest đều không nhận `NaN`.

### 3. Khảo sát biến mục tiêu `target`

In [4]:
target_summary = (
    df["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="Số lượng")
)
target_summary["Tỷ lệ (%)"] = (
    target_summary["Số lượng"] / len(df) * 100
).round(2)

assert set(df["target"].dropna().unique()).issubset({0, 1}), (
    "target chỉ được chứa hai lớp 0 và 1."
)
display(target_summary)


,target,Số lượng,Tỷ lệ (%)
0,0,280462,91.9
1,1,24719,8.1


**Nhận xét:**
- Dữ liệu mất cân bằng nặng: 91,9% trả được nợ (280.462 hồ sơ) so với 8,1% nợ xấu (24.719 hồ sơ).
- Một mô hình đoán bừa "tất cả đều trả được" vẫn đạt Accuracy 91,9% mà không bắt được hồ sơ xấu nào.
- **Đề xuất:** không dùng Accuracy làm chỉ số đánh giá; ưu tiên ROC-AUC, PR-AUC và Recall của nhóm nợ xấu, đồng thời chia tập có phân tầng (`stratify`) ở Mục III.2.

## III. Chuẩn bị dữ liệu

### 1. Xác định Feature và Target

In [5]:
ID_COLUMN = "sk_id_curr"
TARGET_COLUMN = "target"

excluded_columns = [ID_COLUMN, TARGET_COLUMN]
X = df.drop(columns=excluded_columns)
y = df[TARGET_COLUMN].copy()

assert ID_COLUMN not in X.columns, "sk_id_curr không được đưa vào mô hình."
assert TARGET_COLUMN not in X.columns, "target không được đưa vào feature."
assert len(X) == len(y), "Feature và target phải có cùng số dòng."

print(f"Số feature ban đầu: {X.shape[1]:,}")
print(f"Số quan sát: {len(y):,}")
display(pd.DataFrame({
    "Vai trò": ["Feature (X)", "Target (y)"],
    "Nội dung": [f"{X.shape[1]:,} cột đầu vào", TARGET_COLUMN],
}))


Số feature ban đầu: 155
Số quan sát: 305,181


,Vai trò,Nội dung
0,Feature (X),155 cột đầu vào
1,Target (y),target


**Nhận xét:** Còn lại 155 cột đầu vào. `sk_id_curr` chỉ là mã định danh nên không mang thông tin dự báo, còn `target` là nhãn cần dự đoán — đưa vào `X` sẽ gây rò rỉ nhãn.

**Ý nghĩa:**
- Đây là bước dịch bài toán nghiệp vụ ("hồ sơ này có rủi ro không?") thành bài toán học máy: `X` là những gì mô hình được phép nhìn, `y` là cái mô hình phải đoán ra.
- Loại `target` khỏi `X` để tránh rò rỉ nhãn — khi duyệt hồ sơ mới chưa ai biết khách có vỡ nợ hay không, giữ lại sẽ cho chỉ số đẹp giả tạo mà không báo lỗi.
- Loại `sk_id_curr` vì mô hình cây có thể tách theo ngưỡng mã hồ sơ và sinh ra quy tắc rác kiểu "ID lớn thì rủi ro cao".
- Ba dòng `assert` là chốt an toàn: nếu ai sửa nhầm danh sách cột loại trừ, notebook dừng ngay tại đây thay vì chạy tiếp ra một mô hình sai.

### 2. Chia tập Train / Test

In [6]:
# Chia một bước để có hai tập: Train 80% - Test 20%
# stratify=y giữ tỷ lệ nợ xấu ở hai tập giống hệt bảng gốc
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tổng số dòng của bảng gốc, dùng làm mẫu số khi tính tỷ trọng
tong_dong = len(X)

split_summary = pd.DataFrame({
    "Phần dữ liệu": ["X_train (đặc trưng - huấn luyện)", "y_train (nhãn - huấn luyện)",
                     "X_test (đặc trưng - kiểm tra)", "y_test (nhãn - kiểm tra)"],
    "Số dòng": [len(X_train), len(y_train), len(X_test), len(y_test)],
    "Tỷ trọng (%)": [
        round(len(X_train) / tong_dong * 100, 1),
        round(len(y_train) / tong_dong * 100, 1),
        round(len(X_test) / tong_dong * 100, 1),
        round(len(y_test) / tong_dong * 100, 1),
    ],
    "Tỷ lệ nợ xấu (%)": [
        round(y_train.mean() * 100, 2),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2),
        round(y_test.mean() * 100, 2),
    ],
})
display(split_summary)


,Phần dữ liệu,Số dòng,Tỷ trọng (%),Tỷ lệ nợ xấu (%)
0,X_train (đặc trưng - huấn luyện),244144,80.0,8.1
1,y_train (nhãn - huấn luyện),244144,80.0,8.1
2,X_test (đặc trưng - kiểm tra),61037,20.0,8.1
3,y_test (nhãn - kiểm tra),61037,20.0,8.1


**Nhận xét:**
- Chia phân tầng thành Train 244.144 khách (80%) và Test 61.037 khách (20%); `X` và `y` của cùng một tập luôn khớp số dòng, nên không có khách nào bị lệch nhãn.
- Tỷ lệ nợ xấu ở Train và Test đều bám sát 8,1% của bảng gốc — mô hình học và được chấm điểm trên cùng một mức độ khó.
- **Đề xuất:** chỉ có hai tập nên phải phân vai thật rõ — ngưỡng quyết định được dò bằng cross-validation **ngay trong tập Train** ở Mục VIII.2, tập Test để dành chấm điểm cuối ở Mục VIII.3. Riêng phần so sánh ba mô hình ở Mục VII có dùng tập Test, nên những chênh lệch nhỏ giữa các mô hình cần đọc dè dặt.


### 3. Tạo lại Interaction Features sau khi chia tập

#### a. Học ngưỡng phân nhóm từ tập Train

Đoạn code bên dưới học các mốc chia nhóm thu nhập và dư nợ từ tập Train.


In [7]:
# ==== Cac moc va nhan dung de chia nhom ====
# Moc chia nhom tuoi, co dinh theo nghiep vu nen khong phu thuoc du lieu
TUOI_BINS = [0, 25, 35, 45, 55, np.inf]
# Ten goi cua 5 nhom tuoi tuong ung voi cac moc o tren
TUOI_LABELS = ["<25", "25-34", "35-44", "45-54", "55+"]
# Ten goi cua 5 nhom thu nhap, xep tu thap den cao
NHAN_THU_NHAP = ["Rất thấp", "Thấp", "Trung bình", "Cao", "Rất cao"]
# Ten goi cua 4 nhom du no, xep tu thap den cao
NHAN_DU_NO = ["Thấp", "Trung bình", "Cao", "Rất cao"]
# Ba cot cho biet khach da tung tre han o nguon nao chua
TRE_HAN_COLS = ["bureau_max_overdue", "installments_max_late", "pos_cash_max_dpd"]
# Nhan dung cho khach khong co du lieu de xep nhom
NHAN_THIEU = "Không rõ"


def hoc_nguong(cot, so_nhom):
    """Hoc cac moc chia nhom tu mot cot cua tap Train."""
    # qcut chia du lieu thanh cac phan bang nhau va tra ve cac moc cat
    _, moc = pd.qcut(cot, q=so_nhom, retbins=True, duplicates="drop")
    # Ha dau duoi xuong am vo cuc de gia tri nho bat thuong van xep duoc nhom
    moc[0] = -np.inf
    # Nang dau tren len duong vo cuc de gia tri lon bat thuong van xep duoc nhom
    moc[-1] = np.inf
    return moc


# Hoc moc thu nhap: chia khach trong tap Train thanh 5 nhom deu nhau
nguong_thu_nhap = hoc_nguong(X_train["amt_income_total"], 5)
# Danh dau nhung khach con du no de hoc moc rieng cho ho
khach_con_du_no = X_train["bureau_sum_debt"] > 0
# Hoc moc du no: chia rieng nhom con du no thanh 4 muc
nguong_du_no = hoc_nguong(X_train.loc[khach_con_du_no, "bureau_sum_debt"], 4)

# So nhom thuc te co the it hon du kien neu nhieu khach cung mot muc, nen cat nhan cho khop
so_nhom_thu_nhap = len(nguong_thu_nhap) - 1
so_nhom_du_no = len(nguong_du_no) - 1

# In cac moc da hoc de doi chieu khi can giai thich nhom
print("Ngưỡng thu nhập học từ Train:", [f"{v:,.0f}" for v in nguong_thu_nhap[1:-1]])
print("Ngưỡng dư nợ học từ Train:", [f"{v:,.0f}" for v in nguong_du_no[1:-1]])


def tao_interaction(X):
    """Tạo lại hai đặc trưng tương tác từ các mốc học được trên Train."""
    X = X.copy()
    
    # 1. Tương tác Tuổi & Thu nhập (age_income_interaction)
    tuoi_cat = pd.cut(X["age_years"], bins=TUOI_BINS, labels=TUOI_LABELS)
    thu_nhap_cat = pd.cut(X["amt_income_total"], bins=nguong_thu_nhap, labels=NHAN_THU_NHAP)
    X["age_income_interaction"] = tuoi_cat.astype(str) + " | " + thu_nhap_cat.astype(str)
    
    # Gán nhãn "Không rõ" cho những dòng bị khuyết
    khong_ro_mask = X["age_income_interaction"].str.contains("nan", na=True) | X["age_income_interaction"].isna()
    X.loc[khong_ro_mask, "age_income_interaction"] = NHAN_THIEU
    
    # 2. Tương tác Trễ nợ & Dư nợ (late_debt_interaction)
    tre_han = (X[TRE_HAN_COLS] > 0).any(axis=1)
    nhan_tre = np.where(tre_han, "Từng trễ", "Không ghi nhận trễ")
    
    # Phân nhóm dư nợ
    du_no_val = X["bureau_sum_debt"]
    nhan_du_no = pd.cut(du_no_val[du_no_val > 0], bins=nguong_du_no, labels=NHAN_DU_NO).astype(str)
    
    # Tạo danh sách nhãn dư nợ đầy đủ
    nhan_du_no_full = pd.Series(index=X.index, dtype=str)
    nhan_du_no_full.loc[du_no_val == 0] = "Không còn dư nợ"
    nhan_du_no_full.loc[du_no_val > 0] = nhan_du_no
    nhan_du_no_full.loc[du_no_val.isna()] = "Không rõ"
    
    X["late_debt_interaction"] = pd.Series(nhan_tre, index=X.index) + " | " + nhan_du_no_full
    
    # Nếu không có lịch sử nợ ngoài (has_bureau = 0) hoặc bị khuyết thiếu, gán là "Không rõ"
    X.loc[X["has_bureau"] == 0, "late_debt_interaction"] = NHAN_THIEU
    X.loc[X["late_debt_interaction"].str.contains("nan", na=True), "late_debt_interaction"] = NHAN_THIEU
    
    return X


Ngưỡng thu nhập học từ Train: ['99,000', '135,000', '162,000', '225,000']
Ngưỡng dư nợ học từ Train: ['138,889', '394,737', '977,410']


**Nhận xét:**
- Mốc chia nhóm chỉ học từ tập Train, không học từ cả bảng — nếu học từ cả bảng thì thông tin của tập Test đã lọt ngược vào lúc chia nhóm, khiến điểm chấm cuối cao hơn thực tế.
- Mốc dư nợ học riêng trên nhóm khách còn dư nợ; khách đã trả hết được tách thành nhóm riêng ở bước sau thay vì bị dồn chung vào mức thấp nhất.
- **Đề xuất:** khi giải thích một nhóm cho người đọc báo cáo, lấy đúng dãy mốc in ở trên làm căn cứ.


#### b. Tạo đặc trưng tương tác

Đoạn code bên dưới dùng các mốc vừa học để tạo lại hai biến tương tác cho cả hai tập.


In [8]:
# Tạo lại hai biến tương tác cho cả hai tập
X_train = tao_interaction(X_train)
X_test = tao_interaction(X_test)

# Xem thư 5 dòng đầu để kiểm tra nhãn đã ghép đúng chưa
display(X_train[["age_income_interaction", "late_debt_interaction"]].head())


,age_income_interaction,late_debt_interaction
59658,45-54 | Cao,Từng trễ | Không rõ
37171,35-44 | Cao,Từng trễ | Không còn dư nợ
175959,35-44 | Rất cao,Từng trễ | Cao
302179,45-54 | Cao,Từng trễ | Rất cao
124326,25-34 | Thấp,Từng trễ | Cao


**Nhận xét:**
- Mỗi nhãn là hai thông tin ghép lại bằng dấu `|`: `age_income_interaction` ghép nhóm tuổi với nhóm thu nhập, `late_debt_interaction` ghép tình trạng trễ hạn với mức dư nợ. Nhờ vậy mô hình đọc được các tổ hợp đặc thù như "trẻ | thu nhập thấp" hay "từng trễ | dư nợ thấp" như một trường hợp riêng, không phải các yếu tố rời rạc.
- Khách thiếu dữ liệu nhận nhãn `"Không rõ"` chứ không bị bỏ trống — nghĩa là chưa có dữ liệu tại tổ chức khác, không phải lỗi dữ liệu.
- **Đề xuất:** hàm `tao_interaction` dùng lại đúng mốc đã học từ Train nên áp thẳng được cho Test và cho hồ sơ mới ở NB07 mà không phải học lại.


#### c. Kiểm tra và đếm số lượng nhóm trên các tập dữ liệu

Đoạn code bên dưới kiểm tra hai biến vừa tạo và đếm số nhóm trên từng tập.


In [9]:
# Ba bien tuong tac se duoc dung o cac muc sau
interaction_cols = ["age_income_interaction", "late_debt_interaction", "ext_ltv_interaction"]
# Hai bien dau la nhan chu, can kiem tra ky truoc khi ma hoa
nhom_cols = ["age_income_interaction", "late_debt_interaction"]

# Kiem tra tung bien: khong con o trong o bat ky tap nao
for cot in nhom_cols:
    assert not X_train[cot].isna().any(), f"{cot} còn ô trống ở Train"
    assert not X_test[cot].isna().any(), f"{cot} còn ô trống ở Test"

# Kiem tra tung bien: khong co nhom nao chi xuat hien ngoai Train
for cot in nhom_cols:
    nhom_train = set(X_train[cot])
    nhom_la = sorted(set(X_test[cot]) - nhom_train)
    assert not nhom_la, f"{cot} có nhóm chỉ xuất hiện ngoài Train: {nhom_la}"

# Dem so nhom cua tung bien tren tung tap
    bang_nhom = []
    # Dich nghia tieng Viet cho cac bien tuong tac
    ten_tieng_viet = {
        "age_income_interaction": "Tương tác Tuổi & Thu nhập",
        "late_debt_interaction": "Tương tác Trễ nợ & Dư nợ",
        "ext_ltv_interaction": "Tương tác Điểm ngoài & LTV"
    }
for cot in interaction_cols:
    # Dem so dong mang nhan "Khong ro" o tap Train
    so_khong_ro = X_train[cot].astype("string").str.contains(NHAN_THIEU, na=False).sum()
    bang_nhom.append({
            "Biến tương tác": f"{cot} ({ten_tieng_viet.get(cot, '')})",
        "Số nhóm (Train)": X_train[cot].nunique(),
        "Số nhóm (Test)": X_test[cot].nunique(),
        "Số dòng Không rõ (Train)": int(so_khong_ro),
    })
display(pd.DataFrame(bang_nhom))


,Biến tương tác,Số nhóm (Train),Số nhóm (Test),Số dòng Không rõ (Train)
0,age_income_interaction (Tương tác Tuổi & Thu n...,25,25,0
1,late_debt_interaction (Tương tác Trễ nợ & Dư nợ),13,13,40913
2,ext_ltv_interaction (Tương tác Điểm ngoài & LTV),243028,60934,0


**Nhận xét:**
- Hai lệnh `assert` chạy qua không báo lỗi nghĩa là không nhãn nào bị bỏ trống, và tập Test không sinh nhóm lạ nào mà Train chưa từng thấy — điều kiện để bước mã hóa ở Mục III.5 không phải bỏ dòng nào.
- `age_income_interaction` có 25 nhóm (5 nhóm tuổi × 5 nhóm thu nhập), `late_debt_interaction` có 13 nhóm (tổ hợp giữa tình trạng trễ hạn và mức dư nợ, tính cả nhãn `Không rõ`); cả Train và Test đều có đủ.
- **Đề xuất:** nếu sau này đổi `random_state` hoặc tỷ lệ chia mà `assert` báo nhóm lạ, xử lý bằng cách gộp nhóm hiếm chứ đừng bỏ dòng.


### 4. Xử lý giá trị thiếu

#### a. Phân loại cột dữ liệu và định nghĩa quy tắc nghiệp vụ

Tách danh sách cột số (`num_cols`) và cột chữ (`cat_cols`) dựa trên tập Train. Định nghĩa từ điển `COT_KHOI_LUONG` gồm các đặc trưng (đếm, hạn mức, dư nợ, số ngày trễ) cần điền giá trị 0 khi khách hàng không có lịch sử giao dịch tương ứng (xác định qua cờ `has_* = 0` tạo từ Notebook 05).

- **Lý do điền 0:** Nếu khách hàng không có lịch sử nợ ngoài (`has_bureau = 0`), không trả góp (`has_installments = 0`), hay không dùng thẻ tín dụng (`has_credit_card = 0`) thì các số liệu dư nợ, hạn mức, số kỳ và số ngày quá hạn thực tế đúng bằng 0. Điền 0 phản ánh đúng bản chất nghiệp vụ và tránh áp đặt giá trị trung bình sai lệch lên họ.

In [10]:
# Tách danh sách cột số và cột chữ dựa trên tập Train
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

# Định nghĩa các cột đếm, số tiền và số ngày trễ tương ứng với cờ lịch sử
COT_KHOI_LUONG = {
    "has_bureau": [
        "bureau_count", "bureau_sum_credit", "bureau_sum_debt", "bureau_max_overdue",
        "bureau_balance_month_count", "bureau_balance_dpd_month_count",
        "bureau_balance_closed_month_count", "bureau_balance_unknown_month_count",
        "bureau_balance_delinquent_loan_count", "bureau_balance_max_dpd_status",
    ],
    "has_previous": ["previous_count", "previous_sum_credit", "previous_avg_credit"],
    "has_installments": [
        "installments_count", "installments_sum_due", "installments_sum_paid",
        "installments_avg_late", "installments_max_late",
    ],
    "has_pos_cash": ["pos_cash_count", "pos_cash_avg_dpd", "pos_cash_max_dpd"],
    "has_credit_card": [
        "credit_card_count", "credit_card_avg_balance", "credit_card_max_balance",
        "credit_card_avg_limit", "credit_card_max_dpd",
    ],
}


#### b. Điền giá trị 0 cho các cột không có lịch sử giao dịch

Định nghĩa và áp dụng hàm `dien_0_lich_su` để gán giá trị 0 cho các đặc trưng liên quan khi khách hàng không phát sinh giao dịch ở các phân hệ tương ứng. Đồng thời thống kê số lượng ô trống trước và sau bước này.

In [11]:
def dien_0_lich_su(X):
    """Điền 0 cho khách không có lịch sử tương ứng."""
    # Làm việc trên bản sao để không sửa trực tiếp bảng gốc
    X = X.copy()
    # Xét lần lượt từng cờ và nhóm cột đi kèm
    for co, cac_cot in COT_KHOI_LUONG.items():
        # Đánh dấu những khách không có lịch sử này
        khong_co = X[co] == 0
        # Chỉ điền 0 cho đúng những dòng đó, các dòng khác giữ nguyên
        X.loc[khong_co, cac_cot] = X.loc[khong_co, cac_cot].fillna(0)
    return X

# Đếm số ô trống trước khi điền để đối chiếu
thieu_ban_dau = []
for X in (X_train, X_test):
    thieu_ban_dau.append(int(X[num_cols].isna().sum().sum()))

# Điền 0 cho khách không có lịch sử ở cả hai tập
X_train = dien_0_lich_su(X_train)
X_test = dien_0_lich_su(X_test)

# Đếm số ô trống còn lại sau khi điền 0
thieu_con_lai = []
for X in (X_train, X_test):
    thieu_con_lai.append(int(X[num_cols].isna().sum().sum()))


#### c. Điền khuyết cột số bằng giá trị trung vị (Median Imputation)

Dùng `SimpleImputer` với chiến lược trung vị (`median`) học từ tập Train để xử lý các ô trống còn lại của cột số (các cột không thể điền bằng 0 do sai lệch ý nghĩa).

- **Các biến áp dụng và lý do dùng Median:**
  - **Thời gian vay gần nhất (`bureau_recency_days`, `previous_recency_days`):** Điền 0 sẽ biến thành "vừa mới vay hôm nay" (tín hiệu rủi ro cao giả tạo). Điền trung vị (ví dụ: ~300 ngày trước) là phương án trung hòa nhất.
  - **Tuổi tài sản (`own_car_age`):** Khách không có xe thì tuổi xe khuyết. Điền 0 sẽ biến thành "sở hữu xe mới tinh". Điền trung vị là an toàn nhất.
  - **Tỷ số sử dụng (`installments_payment_ratio`, `credit_card_utilization`):** Điền 0 sẽ làm mô hình hiểu sai là bùng nợ hoặc dùng thẻ không tiêu tiền. Điền trung vị giúp ổn định phân phối.
- **Ý nghĩa khả năng tinh chỉnh:** Việc tách riêng imputer này giúp dễ dàng chuyển đổi chiến lược sang `mean`, hoặc dùng các imputer học máy thông minh như `KNNImputer`, `IterativeImputer` trong quá trình tối ưu mô hình mà không ảnh hưởng tới bước điền 0 ở trên.

In [12]:
# Trung vị chỉ được học từ tập Train để tránh rò rỉ thông tin
num_imputer = SimpleImputer(strategy="median")

# fit_transform học trung vị từ Train và điền luôn cho Train
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index
)
# transform áp dụng trung vị đã học từ Train sang tập Test
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index
)


#### d. Điền khuyết cột phân loại và kiểm tra tính toàn vẹn

Điền các giá trị thiếu ở cột chữ bằng nhãn hằng số `"Unknown"`. Chạy kiểm tra `assert` để đảm bảo sạch bóng giá trị thiếu và hiển thị bảng so sánh đối chiếu trước/sau.

- **Tác dụng:**
  - **Tránh lỗi crash khi train model (Chốt chặn an toàn):** Logistic Regression và Random Forest của scikit-learn sẽ báo lỗi và dừng chạy ngay lập tức nếu dữ liệu đầu vào chứa bất kỳ giá trị `NaN` nào. Bước `assert not any(con_thieu)` đảm bảo an toàn tuyệt đối.
  - **Minh bạch hóa thông tin:** Gán nhãn `"Unknown"` cho cột chữ giúp mô hình coi nhóm khuyết thiếu thông tin này là một nhóm phân loại riêng biệt để đánh giá rủi ro (đôi khi việc thiếu thông tin lại chứa ẩn ý rủi ro).
  - **Giám sát dữ liệu:** In bảng đối chiếu so sánh rõ ràng giúp trực quan hóa hiệu quả điền khuyết.

In [13]:
def chuan_hoa_cot_chu(X):
    """Đưa cột chữ về dạng mà SimpleImputer đọc được."""
    # pandas 3.0 dùng kiểu 'string' với ô trống là pd.NA, cần đưa về object và np.nan để SimpleImputer hiểu
    cot_chu = X[cat_cols]
    return cot_chu.astype(object).where(cot_chu.notna(), np.nan)

# Cột chữ thiếu thì điền "Unknown", học và áp dụng từ tập Train
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(chuan_hoa_cot_chu(X_train)), columns=cat_cols, index=X_train.index
)
X_test_cat = pd.DataFrame(
    cat_imputer.transform(chuan_hoa_cot_chu(X_test)), columns=cat_cols, index=X_test.index
)

# Đếm số ô trống còn sót lại sau khi đã điền hết
con_thieu = []
for phan_so, phan_chu in [
    (X_train_num, X_train_cat),
    (X_test_num, X_test_cat),
]:
    con_thieu.append(int(phan_so.isna().sum().sum() + phan_chu.isna().sum().sum()))

# Đảm bảo sạch bóng ô khuyết thiếu trước khi huấn luyện
assert not any(con_thieu), f"Còn giá trị thiếu sau khi điền: {con_thieu}"

# Đếm số ô cột chữ thực sự được điền "Unknown"
so_o_unknown = int((X_train_cat == "Unknown").sum().sum())

print(f"Số cột số: {len(num_cols):,} | Số cột phân loại: {len(cat_cols):,}")
print(f'Số ô cột chữ được điền "Unknown" ở Train: {so_o_unknown:,}')

# Tính số ô được điền 0
so_o_dien_0 = []
for truoc, sau in zip(thieu_ban_dau, thieu_con_lai):
    so_o_dien_0.append(truoc - sau)

# Bảng so sánh số ô trống trước và sau khi điền
display(pd.DataFrame({
    "Tập dữ liệu": ["Train", "Test"],
    "Ô trống ban đầu": thieu_ban_dau,
    "Điền 0 (không có lịch sử)": so_o_dien_0,
    "Điền trung vị": thieu_con_lai,
    "Còn trống sau khi điền": con_thieu,
}))


Số cột số: 136 | Số cột phân loại: 19
Số ô cột chữ được điền "Unknown" ở Train: 0


,Tập dữ liệu,Ô trống ban đầu,Điền 0 (không có lịch sử),Điền trung vị,Còn trống sau khi điền
0,Train,2142589,1364403,778186,0
1,Test,532348,339492,192856,0


**Nhận xét:**
- Tập Train có 2.142.589 ô trống ở cột số. Trong đó 1.364.403 ô (64%) được điền 0 vì đó là khách không có lịch sử tương ứng — không có thẻ tín dụng thì dư nợ thẻ đúng bằng 0, chưa từng vay ở tổ chức khác thì số khoản vay bằng 0. Chỉ những dòng có cờ `has_* = 0` mới được điền 0.
- 778.186 ô còn lại điền bằng trung vị (con số nằm giữa của cột) tính từ tập Train, rồi áp sang tập Test.
- 19 cột chữ không phải điền ô nào: NB03 đã điền sẵn 17 cột, còn `late_debt_interaction` thì Mục III.3 vừa gán nhãn `"Không rõ"`. Phần điền `"Unknown"` ở đây chỉ để phòng xa.


Đoạn code bên dưới liệt kê các cột còn phải dùng trung vị và giá trị đã điền.

In [14]:
# Lay trung vi ma imputer da hoc, gan lai ten cot cho de doc
trung_vi = pd.Series(num_imputer.statistics_, index=num_cols)

# Gom tat ca cot da duoc dien 0 thanh mot danh sach phang
cot_dien_0 = []
for cac_cot in COT_KHOI_LUONG.values():
    cot_dien_0.extend(cac_cot)

# Dem so o trong con lai truoc khi dien trung vi
con_trung_vi = X_train[num_cols].isna().sum()
# Chi giu lai nhung cot thuc su co o trong, sap xep giam dan
con_trung_vi = con_trung_vi[con_trung_vi > 0].sort_values(ascending=False)

# Bang tom tat hai cach dien va y nghia cua tung cach
display(pd.DataFrame({
    "Cách điền": ["Điền 0 khi không có lịch sử", "Trung vị từ Train"],
    "Số cột": [len(cot_dien_0), len(con_trung_vi)],
    "Ý nghĩa": [
        "Cột đếm, số tiền, mức trễ hạn - không có lịch sử nghĩa là bằng 0",
        "Cột thời gian, tỷ số, tuổi xe - điền 0 sẽ tạo ra nghĩa sai",
    ],
}))

# Lay 8 cot dung trung vi nhieu nhat de doi chieu gia tri da dien
top_trung_vi = con_trung_vi.head(8)
display(pd.DataFrame({
    "Cột dùng trung vị": top_trung_vi.index,
    "Ô trống (Train)": top_trung_vi.values,
    "Tỷ lệ (%)": (top_trung_vi.values / len(X_train) * 100).round(1),
    "Trung vị đã điền": trung_vi[top_trung_vi.index].round(2).values,
}))

,Cách điền,Số cột,Ý nghĩa
0,Điền 0 khi không có lịch sử,26,"Cột đếm, số tiền, mức trễ hạn - không có lịch ..."
1,Trung vị từ Train,24,"Cột thời gian, tỷ số, tuổi xe - điền 0 sẽ tạo ..."


,Cột dùng trung vị,Ô trống (Train),Tỷ lệ (%),Trung vị đã điền
0,credit_card_utilization,175497,71.9,0.26
1,own_car_age,161096,66.0,9.00
2,bureau_debt_ratio,41679,17.1,0.22
3,bureau_avg_days_credit,35047,14.4,-1051.75
4,bureau_latest_days_credit,35047,14.4,-300.00
5,bureau_recency_days,35047,14.4,300.00
6,amt_req_credit_bureau_day,33101,13.6,0.00
7,amt_req_credit_bureau_hour,33101,13.6,0.00


**Nhận xét:**
- 26 cột đếm/tiền/mức trễ được điền 0; 24 cột còn lại vẫn dùng trung vị vì điền 0 vào chúng sẽ tạo nghĩa sai. Ví dụ `bureau_recency_days` là "số ngày kể từ lần vay gần nhất" (trung vị 300 ngày) — điền 0 sẽ biến khách chưa từng vay thành khách vừa vay hôm nay, tức bịa ra tín hiệu rủi ro ngược hẳn.
- `own_car_age` vẫn phải dùng trung vị 9 năm cho 161.096 khách (66,0% tập Train) không có ô tô, vì bảng không có cột cờ dạng số cho việc có xe; thông tin này nằm ở cột chữ `flag_own_car` nên mô hình vẫn đọc được.
- **Đề xuất:** khi thêm feature mới ở các notebook sau, luôn tạo kèm một cờ `has_*` như NB05 đã làm — có cờ thì mới tách được "khách không có" khỏi "thiếu dữ liệu" ở bước này.


### 5. Mã hóa dữ liệu

Đoạn code bên dưới đổi 19 cột chữ thành cột số bằng one-hot, danh sách nhóm học từ tập Train.

In [15]:
# Encoder đổi mỗi nhóm chữ thành một cột nhận giá trị 0 hoặc 1
# handle_unknown="ignore": gặp nhóm lạ ở Test thì cho cả dòng đó bằng 0 thay vì báo lỗi
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
# Danh sách nhóm chỉ được học từ tập Train
encoder.fit(X_train_cat)
# Lấy tên của các cột nhị phân vừa sinh ra
encoded_cols = encoder.get_feature_names_out(cat_cols)

def build_matrix(df_num, df_cat, index):
    """Ghép phần cột số với phần cột chữ đã mã hóa."""
    # Đổi phần chữ thành các cột 0/1 theo danh sách nhóm đã học
    encoded = pd.DataFrame(encoder.transform(df_cat), columns=encoded_cols, index=index)
    # Nối hai phần lại theo chiều ngang, giữ nguyên thứ tự dòng
    return pd.concat([df_num, encoded], axis=1)

# Dựng ma trận đầu vào hoàn chỉnh cho cả hai tập
X_train_prep = build_matrix(X_train_num, X_train_cat, X_train.index)
X_test_prep = build_matrix(X_test_num, X_test_cat, X_test.index)

# Chốt danh sách tên cột để các mục sau và NB07 dùng chung
FEATURE_NAMES = X_train_prep.columns.tolist()

# Lệch thứ tự cột là mô hình đọc nhầm cột mà không báo lỗi, nên phải chốt cả thứ tự
assert list(X_test_prep.columns) == FEATURE_NAMES, "Test lệch cột so với Train"
# Đến bước này không còn ô trống nào
assert not X_train_prep.isna().any().any(), "Ma trận Train còn ô trống"
# Toàn bộ cột phải là kiểu số thì mô hình mới đọc được
assert X_train_prep.select_dtypes(exclude="number").empty, "Còn cột chưa phải kiểu số"

# Bảng so sánh số cột trước và sau khi mã hóa
display(pd.DataFrame({
    "Tập dữ liệu": ["Train", "Test"],
    "Số dòng": [len(X_train_prep), len(X_test_prep)],
    "Số cột trước mã hóa": [len(num_cols) + len(cat_cols)] * 2,
    "Số cột sau mã hóa": [X_train_prep.shape[1], X_test_prep.shape[1]],
}))


,Tập dữ liệu,Số dòng,Số cột trước mã hóa,Số cột sau mã hóa
0,Train,244144,155,320
1,Test,61037,155,320


**Nhận xét:**
- 19 cột chữ biến thành 184 cột nhị phân — mỗi cột là một câu hỏi có/không, ví dụ "khách có làm ở `Business Entity Type 3` không". Tổng số cột đầu vào tăng từ 155 lên 320.
- Cả hai tập đều đúng 320 cột và **cùng thứ tự cột**, nên mô hình huấn luyện trên Train đọc đúng cột khi dự đoán cho Test.
- Danh sách nhóm chỉ học từ Train. `handle_unknown="ignore"` nghĩa là gặp nhóm lạ ở Test thì dòng đó nhận toàn số 0 thay vì báo lỗi — Mục III.3 đã kiểm tra không có nhóm lạ nên đây chỉ là lưới an toàn.


Đoạn code bên dưới xem cột chữ nào sinh ra nhiều cột nhị phân nhất.

In [16]:
# So nhom cua mot cot chu chinh la so cot nhi phan no sinh ra
so_nhom = {}
for ten_cot, cac_nhom in zip(cat_cols, encoder.categories_):
    so_nhom[ten_cot] = len(cac_nhom)

# Doi sang Series va sap xep giam dan de xem cot nao sinh nhieu cot nhat
so_nhom = pd.Series(so_nhom).sort_values(ascending=False)
top_nhom = so_nhom.head(8)

display(pd.DataFrame({
    "Cột chữ": top_nhom.index,
    "Số cột nhị phân sinh ra": top_nhom.values,
    f"Tỷ trọng trong {len(encoded_cols)} cột (%)": (top_nhom.values / len(encoded_cols) * 100).round(1),
}))

# Dem ty le o mang gia tri 0 trong phan ma hoa
ty_le_o_0 = (X_train_prep[list(encoded_cols)] == 0).mean().mean()
print(f"Tỷ lệ ô mang giá trị 0 trong phần mã hóa: {ty_le_o_0:.1%}")

,Cột chữ,Số cột nhị phân sinh ra,Tỷ trọng trong 184 cột (%)
0,organization_type,58,31.5
1,age_income_interaction,25,13.6
2,occupation_type,19,10.3
3,late_debt_interaction,13,7.1
4,wallsmaterial_mode,8,4.3
5,name_income_type,8,4.3
6,weekday_appr_process_start,7,3.8
7,name_type_suite,7,3.8


Tỷ lệ ô mang giá trị 0 trong phần mã hóa: 89.7%


**Nhận xét:**
- `organization_type` là loại tổ chức nơi khách làm việc, có 58 loại khác nhau (`Business Entity Type 3`, `Self-employed`, `Medicine`...). Mỗi loại thành một cột nên riêng nó sinh 58 cột, chiếm 31,5% toàn bộ phần mã hóa.
- 17 trong 58 loại đó rất ít khách — dưới 500 người mỗi loại, ít nhất là `Industry: type 8` chỉ có 24 người trên toàn bộ dữ liệu. Vẫn giữ nguyên chứ không gộp chúng lại, vì gộp thì mất tên tổ chức cụ thể khi giải thích kết quả cho người dùng nghiệp vụ.
- 89,7% ô trong phần mã hóa mang giá trị 0, vì mỗi khách chỉ thuộc đúng một loại tổ chức, một nghề, một tình trạng hôn nhân — các cột còn lại của khách đó đều bằng 0.
- **Đề xuất:** khi đọc hệ số Logistic Regression ở Mục IV, bỏ qua cột của 17 loại tổ chức hiếm này — hệ số tính từ vài chục người nên không đủ tin cậy để kết luận.


### 6. Chuẩn hóa dữ liệu khi cần

Đoạn code bên dưới đưa các cột số về cùng thang đo, học trung bình và độ lệch chuẩn từ tập Train.

In [17]:
# Chỉ chuẩn hóa 136 cột số gốc; 184 cột nhị phân 0/1 giữ nguyên vì đã cùng thang đo
cot_chuan_hoa = num_cols

# Scaler chỉ học trung bình và độ lệch chuẩn từ tập Train
scaler = StandardScaler()
scaler.fit(X_train_prep[cot_chuan_hoa])

def chuan_hoa(X):
    """Chuẩn hóa phần cột số, giữ nguyên phần cột nhị phân."""
    # Làm việc trên bản sao để không sửa ma trận gốc
    X = X.copy()
    # Thay phần cột số bằng bản đã chuẩn hóa, các cột còn lại không động đến
    X[cot_chuan_hoa] = scaler.transform(X[cot_chuan_hoa])
    return X

# Bản đã chuẩn hóa dành cho Logistic Regression ở Mục IV
X_train_scaled = chuan_hoa(X_train_prep)
X_test_scaled = chuan_hoa(X_test_prep)

# Cột hằng số (mọi khách đều một giá trị) không thể có độ lệch chuẩn 1, nên tách ra khi kiểm tra
do_lech_goc = X_train_prep[cot_chuan_hoa].std()
cot_hang_so = do_lech_goc[do_lech_goc == 0].index.tolist()
cot_binh_thuong = [c for c in cot_chuan_hoa if c not in cot_hang_so]

# Kiểm tra: cột số sau chuẩn hóa phải có trung bình xấp xỉ 0
trung_binh_sau = X_train_scaled[cot_chuan_hoa].mean().abs().max()
assert trung_binh_sau < 1e-6, f"Trung bình sau chuẩn hóa chưa về 0: {trung_binh_sau}"

# Kiểm tra: cột số sau chuẩn hóa phải có độ lệch chuẩn xấp xỉ 1
do_lech_sau = X_train_scaled[cot_binh_thuong].std()
assert do_lech_sau.between(0.99, 1.01).all(), "Độ lệch chuẩn sau chuẩn hóa chưa về 1"

# Kiểm tra: phần nhị phân vẫn chỉ chứa 0 và 1, tức không bị chuẩn hóa nhầm
gia_tri_nhi_phan = set(pd.unique(X_train_scaled[list(encoded_cols)].values.ravel()))
assert gia_tri_nhi_phan <= {0.0, 1.0}, "Cột nhị phân đã bị chuẩn hóa ngoài ý muốn"

print(f"Số cột được chuẩn hóa: {len(cot_chuan_hoa):,}")
print(f"Số cột nhị phân giữ nguyên: {len(encoded_cols):,}")
print(f"Cột hằng số không mang thông tin: {cot_hang_so}")


Số cột được chuẩn hóa: 136
Số cột nhị phân giữ nguyên: 184
Cột hằng số không mang thông tin: ['flag_mobil']


**Nhận xét:**
- Chuẩn hóa ở đây nghĩa là **đổi cách ghi con số**: thay vì ghi "khách này vay 987.117", ta ghi "khách này vay cao hơn mức trung bình đúng 1 mức chênh lệch thông thường", và viết gọn thành `1,0`. Khách vay đúng mức trung bình 596.239 thì ghi thành `0`. Sau khi đổi, mọi cột số đều chạy quanh khoảng −3 đến +3, không còn cột nào chạy tới hàng triệu.
- 184 cột nhị phân giữ nguyên `0/1` vì chúng chỉ trả lời có hoặc không, không có đơn vị nào để đổi.
- Mức trung bình và mức chênh lệch dùng để đổi chỉ tính trên tập Train rồi áp sang tập Test — cùng nguyên tắc với mốc chia nhóm ở Mục III.3, cách điền ở Mục III.4 và danh sách nhóm ở Mục III.5.
- `flag_mobil` là cột mà mọi khách đều mang cùng một giá trị, nên sau khi đổi tất cả đều thành 0. Cột này không giúp phân biệt khách nào với khách nào, mô hình sẽ tự bỏ qua.


Đoạn code bên dưới đối chiếu vài cột tiêu biểu trước và sau khi chuẩn hóa.

In [18]:
# Chon vai cot co thang do rat khac nhau de thay ro tac dung
cot_vi_du = ["amt_credit", "bureau_recency_days", "age_years", "ext_sources_mean"]

# Gom so lieu truoc va sau chuan hoa cua tung cot
bang_doi_chieu = []
for cot in cot_vi_du:
    bang_doi_chieu.append({
        "Cột": cot,
        "Trung bình trước": round(float(X_train_prep[cot].mean()), 2),
        "Độ lệch chuẩn trước": round(float(X_train_prep[cot].std()), 2),
        "Trung bình sau": round(float(X_train_scaled[cot].mean()), 4),
        "Độ lệch chuẩn sau": round(float(X_train_scaled[cot].std()), 4),
    })
display(pd.DataFrame(bang_doi_chieu))

# Tim nhom hiem nhat trong phan nhi phan de thay vi sao khong chuan hoa phan nay
ty_le_nhom = X_train_prep[list(encoded_cols)].mean().sort_values()
ten_nhom_hiem = ty_le_nhom.index[0]
ty_le_hiem = ty_le_nhom.iloc[0]
# Cong thuc chuan hoa cho cot 0/1: gia tri 1 se thanh (1 - p) / can(p * (1 - p))
gia_tri_neu_chuan_hoa = (1 - ty_le_hiem) / np.sqrt(ty_le_hiem * (1 - ty_le_hiem))

print(f"Nhóm hiếm nhất: {ten_nhom_hiem}")
print(f"Số khách thuộc nhóm này ở Train: {int(ty_le_hiem * len(X_train_prep)):,}")
print(f"Nếu chuẩn hóa cột này, giá trị của họ sẽ thành: {gia_tri_neu_chuan_hoa:,.1f}")

,Cột,Trung bình trước,Độ lệch chuẩn trước,Trung bình sau,Độ lệch chuẩn sau
0,amt_credit,596239.26,390877.48,-0.0,1.0
1,bureau_recency_days,461.74,501.60,-0.0,1.0
2,age_years,43.92,11.94,-0.0,1.0
3,ext_sources_mean,0.51,0.11,0.0,1.0


Nhóm hiếm nhất: name_income_type_Maternity leave
Số khách thuộc nhóm này ở Train: 4
Nếu chuẩn hóa cột này, giá trị của họ sẽ thành: 247.1


**Nhận xét:**
- Trước khi đổi, `amt_credit` tính bằng đồng nên chạy tới hàng triệu, còn `ext_sources_mean` là điểm đánh giá chỉ chạy từ 0,02 đến 0,85. Logistic Regression làm việc bằng cách nhân mỗi cột với một trọng số rồi cộng lại, nên khi một cột lớn gấp hàng triệu lần cột kia thì nó lấn át toàn bộ phép cộng. Sau khi đổi, cả bốn cột trong bảng đều về cùng khoảng giá trị nên không cột nào lấn cột nào.
- Đó cũng là lý do **không** đổi đơn vị cho 184 cột nhị phân: nhóm `name_income_type_Maternity leave` chỉ có 4 khách trong 244.144 dòng Train. Nếu đổi, 4 người này nhận giá trị 247,1 còn tất cả những người khác xấp xỉ 0 — mô hình sẽ tưởng 4 người đó là trường hợp cực kỳ đặc biệt và dồn sự chú ý vào họ, trong khi thực tế chỉ là một nhóm quá ít người.
- **Đề xuất:** khi lưu bộ tiền xử lý ở Mục VIII.4, lưu kèm danh sách 136 cột đã đổi đơn vị. Nếu NB07 đổi nhầm cả 320 cột thì kết quả dự đoán sai lệch mà không có thông báo lỗi nào.


## IV. Mô hình Logistic Regression

### 1. Huấn luyện mô hình

Đoạn code bên dưới huấn luyện Logistic Regression trên tập Train đã đổi đơn vị ở Mục III.6.

In [19]:
# Logistic Regression can du lieu da doi don vi nen dung ban X_train_scaled
# class_weight="balanced": no xau chi chiem 8,1% nen moi khach no xau duoc tinh nang hon,
# neu khong mo hinh chi can doan "ai cung tra duoc no" la da dung 91,9%
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)

# Do thoi gian huan luyen de doi chieu voi hai mo hinh sau
bat_dau = time.time()
lr_model.fit(X_train_scaled, y_train)
thoi_gian_lr = time.time() - bat_dau

# So vong lap thuc te cho biet mo hinh da tim xong trong so hay bi cat giua chung
so_vong_lap = int(lr_model.n_iter_[0])

# Tinh xem mot khach no xau dang duoc tinh nang gap bao nhieu lan khach tra duoc no
ty_le_can_bang = (1 - y_train.mean()) / y_train.mean()

print(f"Thời gian huấn luyện: {thoi_gian_lr:.1f} giây")
print(f"Số vòng lặp: {so_vong_lap} / 1000")
print(f"Một khách nợ xấu được tính nặng gấp: {ty_le_can_bang:.1f} lần")

# Cham dung 1000 vong nghia la bi cat giua chung, ket qua khong dang tin
assert so_vong_lap < 1000, "Mô hình chưa hội tụ, cần tăng max_iter"

Thời gian huấn luyện: 6.8 giây
Số vòng lặp: 139 / 1000
Một khách nợ xấu được tính nặng gấp: 11.3 lần


**Nhận xét:**
- Mô hình dừng sau 139 vòng lặp trong giới hạn 1.000, tức nó tự tìm xong bộ trọng số rồi dừng chứ không bị cắt ngang. Nếu số vòng chạm đúng 1.000 thì nghĩa là còn đang dò dở, kết quả không dùng được — dòng `assert` cuối cell chặn sẵn trường hợp đó.
- Nợ xấu chỉ chiếm 8,1% nên `class_weight="balanced"` bảo mô hình tính mỗi khách nợ xấu nặng gấp 11,3 lần khách trả được nợ. Không có nó, cách "an toàn" nhất với mô hình là đoán ai cũng trả được nợ — đúng 91,9% số trường hợp nhưng không bắt được hồ sơ rủi ro nào.


### 2. Dự đoán và đánh giá trên tập Test

Đoạn code bên dưới định nghĩa hàm chấm điểm dùng chung cho cả ba mô hình, rồi chấm cho Logistic Regression.

In [20]:
# Nhan tieng Viet dung chung cho moi bao cao va bieu do trong notebook
CLASS_LABELS = ["Trả được nợ (0)", "Nợ xấu (1)"]


def danh_gia(ten_mo_hinh, y_true, y_proba, nguong=0.5, mau="Blues"):
    """Cham diem mot mo hinh: tra ve cac chi so, in bao cao va ve ma tran nham lan."""
    # Doi xac suat thanh quyet dinh: tren nguong thi coi la no xau
    y_pred = (y_proba >= nguong).astype(int)

    # Nam chi so dung de so sanh ba mo hinh o Muc VII
    chi_so = {
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "PR-AUC": average_precision_score(y_true, y_proba),
        "F1-Score": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
    }

    print(f"{ten_mo_hinh} | ROC-AUC: {chi_so['ROC-AUC']:.4f} | PR-AUC: {chi_so['PR-AUC']:.4f}")
    print(f"\nBáo cáo phân loại (ngưỡng {nguong:.4f}):")
    print(classification_report(y_true, y_pred, target_names=CLASS_LABELS))

    # Ma tran nham lan cho thay bat dung bao nhieu va canh bao oan bao nhieu
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap=mau, cbar=False,
        xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS,
    )
    plt.xlabel("Nhãn dự đoán")
    plt.ylabel("Nhãn thực tế")
    plt.title(f"Ma trận nhầm lẫn (Confusion Matrix) — {ten_mo_hinh}, ngưỡng {nguong:.4f}")
    plt.tight_layout()
    plt.show()
    return chi_so


# Lay xac suat no xau ma mo hinh cham cho tung khach o tap Test
y_test_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Cham diem tren Test de so sanh ba mo hinh; rieng nguong quyet dinh
# duoc do bang cross-validation trong tap Train o Muc VIII.2
chi_so_lr_test = danh_gia("Logistic Regression", y_test, y_test_proba_lr, mau="Blues")

# Ty le no xau that la moc co so de doi chieu PR-AUC
print(f"Tỷ lệ nợ xấu thật ở Test: {y_test.mean():.4f}")

Logistic Regression | ROC-AUC: 0.7616 | PR-AUC: 0.2401

Báo cáo phân loại (ngưỡng 0.5000):
                 precision    recall  f1-score   support

Trả được nợ (0)       0.96      0.70      0.81     56093
     Nợ xấu (1)       0.17      0.69      0.27      4944

       accuracy                           0.70     61037
      macro avg       0.57      0.69      0.54     61037
   weighted avg       0.90      0.70      0.77     61037



Tỷ lệ nợ xấu thật ở Test: 0.0810


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\2189944949.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- ROC-AUC 0,7616 nghĩa là: lấy ngẫu nhiên một khách thực sự vỡ nợ và một khách thực sự trả được nợ, mô hình chấm điểm rủi ro cho người vỡ nợ cao hơn trong khoảng 76,2% số lần. Đoán mò sẽ là 50%.
- PR-AUC 0,2401 so với mốc cơ sở 0,0810 (đúng bằng tỷ lệ nợ xấu thật) — cao hơn 2,96 lần mức đoán mò, nhưng con số tuyệt đối vẫn thấp vì tìm nhóm thiểu số 8% là bài toán khó.
- Ở ngưỡng 0,5, mô hình bắt đúng 3.387 trong 4.944 khách nợ xấu (Recall 0,6851) nhưng đổi lại cảnh báo oan khoảng 16.700 khách tốt. Nói cách khác, cứ 100 hồ sơ bị cảnh báo thì chỉ gần 17 hồ sơ vỡ nợ thật (Precision 0,1684), hơn 83 hồ sơ còn lại bị từ chối oan.
- **Đề xuất:** ngưỡng 0,5 chỉ là mức mặc định chứ chưa phải lựa chọn có cân nhắc. Mục VIII.2 sẽ dò lại ngưỡng bằng cross-validation trong tập Train để cân bằng giữa việc bắt sót và việc từ chối oan.


### 3. Hệ số của mô hình

Đoạn code bên dưới xem cột nào đẩy rủi ro lên cao nhất và cột nào kéo rủi ro xuống thấp nhất.

In [21]:
# Moi cot dau vao co mot he so: duong la tang rui ro, am la giam rui ro
he_so = pd.Series(lr_model.coef_[0], index=FEATURE_NAMES).sort_values()


def dem_khach_trong_nhom(ten_cot):
    """Dem so khach Train thuoc nhom nay; cot so thi khong co khai niem nhom."""
    # Cot nhi phan chi nhan 0 hoac 1 nen dem duoc so khach mang gia tri 1
    if ten_cot in list(encoded_cols):
        return f"{int((X_train_prep[ten_cot] == 1).sum()):,}"
    # Cot so thi moi khach deu co gia tri nen khong dem theo nhom
    return "(cột số)"


# Gom 10 he so am nhat va 10 he so duong nhat vao mot bang
bang_he_so = []
for ten_cot in list(he_so.head(10).index) + list(he_so.tail(10).index):
    bang_he_so.append({
        "Cột": ten_cot,
        "Hệ số": round(float(he_so[ten_cot]), 4),
        "Chiều tác động": "Giảm rủi ro" if he_so[ten_cot] < 0 else "Tăng rủi ro",
        "Số khách thuộc nhóm (Train)": dem_khach_trong_nhom(ten_cot),
    })
display(pd.DataFrame(bang_he_so))

# Dem so cot gan nhu khong anh huong gi den ket qua
so_cot_gan_0 = int((he_so.abs() < 0.01).sum())
print(f"Số cột có hệ số gần 0 (|hệ số| < 0,01): {so_cot_gan_0:,} / {len(he_so):,}")

,Cột,Hệ số,Chiều tác động,Số khách thuộc nhóm (Train)
0,organization_type_Industry: type 12,-0.4838,Giảm rủi ro,282
1,name_education_type_Academic degree,-0.4432,Giảm rủi ro,139
2,name_income_type_Pensioner,-0.4181,Giảm rủi ro,"43,984"
3,age_income_interaction_55+ | Rất thấp,-0.3987,Giảm rủi ro,"16,478"
4,organization_type_Security Ministries,-0.3458,Giảm rủi ro,"1,562"
5,age_income_interaction_<25 | Rất cao,-0.3395,Giảm rủi ro,687
6,age_income_interaction_55+ | Thấp,-0.3356,Giảm rủi ro,"15,177"
7,age_income_interaction_45-54 | Rất thấp,-0.3329,Giảm rủi ro,"10,435"
8,organization_type_Military,-0.3297,Giảm rủi ro,"2,095"
9,organization_type_Trade: type 6,-0.3277,Giảm rủi ro,509


Số cột có hệ số gần 0 (|hệ số| < 0,01): 38 / 320


**Nhận xét:**
- Các hệ số dương lớn nhất đều thuộc nhóm rất ít khách: `Realtor` +0,9129 (**317 khách**), `Legal Services` +0,5076 (**250 khách**), `Unemployed` +0,3720 (**13 khách**/244.144). Học từ vài trăm người thì không suy ra được quy luật.
- Các nhóm đông khách cho kết quả đáng tin hơn và khớp với NB04: về hưu −0,4181 (43.984 khách) và trên 55 tuổi thu nhập rất thấp −0,3987 (16.478 khách) giảm rủi ro; 35–44 tuổi thu nhập trung bình +0,3766 (7.722 khách) và 35–44 tuổi thu nhập cao +0,2680 (17.966 khách) tăng rủi ro.
- 38/320 cột có hệ số gần 0, tức gần như không ảnh hưởng tới quyết định của mô hình.
- **Đề xuất:** trích bảng này vào báo cáo thì phải kèm cột số khách, và chỉ diễn giải các dòng có từ vài nghìn khách trở lên.


## V. Mô hình Random Forest

### 1. Huấn luyện mô hình

Đoạn code bên dưới huấn luyện Random Forest trên tập Train chưa đổi đơn vị ở Mục III.4.


In [22]:
# Random Forest cat du lieu theo nguong lon hon / nho hon nen khong can doi don vi,
# vi vay dung ban goc X_train_prep chu khong dung ban X_train_scaled cua Logistic Regression
rf_model = RandomForestClassifier(
    n_estimators=300,        # trong 300 cay, moi cay hoc mot phan du lieu roi lay y kien chung
    max_depth=12,            # moi cay chi hoi toi da 12 cau, khong duoc hoi sau hon
    min_samples_leaf=50,     # moi ket luan cuoi cung phai dua tren it nhat 50 khach
    class_weight="balanced", # tinh nang khach no xau, giong Logistic Regression o Muc IV.1
    n_jobs=-1,               # dung het cac nhan CPU cho nhanh
    random_state=42,
)

# Do thoi gian huan luyen de doi chieu voi Logistic Regression o Muc VII
bat_dau = time.time()
rf_model.fit(X_train_prep, y_train)
thoi_gian_rf = time.time() - bat_dau

# Do sau thuc te cua cac cay: neu tat ca deu cham tran 12 thi hang rao dang that su co tac dung
do_sau_cac_cay = [cay.get_depth() for cay in rf_model.estimators_]

# Dem so ket luan cuoi cung (la) trung binh moi cay dua ra
so_la_trung_binh = np.mean([cay.get_n_leaves() for cay in rf_model.estimators_])

print(f"Thời gian huấn luyện: {thoi_gian_rf:.1f} giây (Logistic Regression: {thoi_gian_lr:.1f} giây)")
print(f"Số cây: {len(rf_model.estimators_):,}")
print(f"Độ sâu các cây — nhỏ nhất {min(do_sau_cac_cay)}, lớn nhất {max(do_sau_cac_cay)} (giới hạn 12)")
print(f"Số kết luận cuối cùng trung bình mỗi cây: {so_la_trung_binh:,.0f}")

# Thieu cay so voi khai bao nghia la co gi do sai trong qua trinh huan luyen
assert len(rf_model.estimators_) == 300, "Số cây thực tế không khớp với n_estimators"


Thời gian huấn luyện: 29.5 giây (Logistic Regression: 6.8 giây)
Số cây: 300
Độ sâu các cây — nhỏ nhất 12, lớn nhất 12 (giới hạn 12)
Số kết luận cuối cùng trung bình mỗi cây: 896


**Nhận xét:**
- Random Forest dùng bản dữ liệu gốc chứ không dùng bản đã đổi đơn vị ở Mục III.6. Mô hình này ra quyết định bằng câu hỏi dạng "khoản vay có trên 600.000 không" — đổi đơn vị thì thứ tự khách vẫn y nguyên nên câu trả lời không đổi. Chuẩn hóa chỉ cần cho Logistic Regression vì nó nhân từng cột với trọng số rồi cộng lại.
- **Cả 300 cây đều dừng đúng ở độ sâu 12**, tức cây nào cũng còn muốn hỏi thêm nhưng bị chặn. Nếu để cây hỏi thoải mái, nó sẽ hỏi tới mức nhớ được từng khách trong tập Train và học vẹt. Con số 12 là do ta chọn chứ chưa phải mức tốt nhất.
- Mỗi cây chia 244.144 khách thành 896 nhóm, trung bình 272 khách một nhóm. Một cây sâu 12 tầng nếu chia hết cỡ sẽ được 4.096 nhóm, ở đây chỉ 896 — nghĩa là `min_samples_leaf=50` cũng đang chặn thật: nhiều nhánh phải dừng sớm vì chia tiếp sẽ còn dưới 50 khách. Đây chính là hàng rào mà Mục IV.3 còn thiếu, nơi mô hình rút ra kết luận từ nhóm chỉ 13 người.
- Huấn luyện mất khoảng nửa phút, so với chưa tới 10 giây của Logistic Regression — chậm hơn chừng 4 lần vì phải dựng 300 cây thay vì tìm một bộ trọng số.


### 2. Dự đoán và đánh giá trên tập Test

Đoạn code bên dưới chấm điểm Random Forest bằng hàm `danh_gia` đã dựng ở Mục IV.2, rồi đặt cạnh Logistic Regression.


In [23]:
# Lay xac suat no xau ma 300 cay cung nhau cham cho tung khach o tap Test
y_test_proba_rf = rf_model.predict_proba(X_test_prep)[:, 1]

# Cham diem tren Test de so sanh ba mo hinh; rieng nguong quyet dinh
# duoc do bang cross-validation trong tap Train o Muc VIII.2
chi_so_rf_test = danh_gia("Random Forest", y_test, y_test_proba_rf, mau="Greens")

# Dat hai mo hinh canh nhau de thay ro hon kem o tung chi so
bang_so_sanh = pd.DataFrame({
    "Logistic Regression": chi_so_lr_test,
    "Random Forest": chi_so_rf_test,
}).round(4)

# Cot chenh lech cho biet Random Forest hon hay kem, do bang diem tuyet doi
bang_so_sanh["Chênh lệch"] = (bang_so_sanh["Random Forest"] - bang_so_sanh["Logistic Regression"]).round(4)
display(bang_so_sanh)

# Doi xac suat thanh quyet dinh o nguong 0,5 de dem so ho so bi tu choi oan
y_test_pred_rf = (y_test_proba_rf >= 0.5).astype(int)
so_bi_canh_bao = int(y_test_pred_rf.sum())
so_canh_bao_dung = int(((y_test_pred_rf == 1) & (y_test == 1)).sum())
so_canh_bao_oan = so_bi_canh_bao - so_canh_bao_dung

print(f"Ở ngưỡng 0,5: cảnh báo {so_bi_canh_bao:,} hồ sơ, trong đó {so_canh_bao_dung:,} vỡ nợ thật và {so_canh_bao_oan:,} bị từ chối oan")
print(f"Số khách nợ xấu bị bỏ sót: {int(((y_test_pred_rf == 0) & (y_test == 1)).sum()):,} / {int(y_test.sum()):,}")

Random Forest | ROC-AUC: 0.7548 | PR-AUC: 0.2344

Báo cáo phân loại (ngưỡng 0.5000):
                 precision    recall  f1-score   support

Trả được nợ (0)       0.96      0.76      0.84     56093
     Nợ xấu (1)       0.18      0.61      0.28      4944

       accuracy                           0.74     61037
      macro avg       0.57      0.68      0.56     61037
   weighted avg       0.89      0.74      0.80     61037



C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\2189944949.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Logistic Regression,Random Forest,Chênh lệch
ROC-AUC,0.7616,0.7548,-0.0068
PR-AUC,0.2401,0.2344,-0.0057
F1-Score,0.2704,0.2793,0.0089
Precision,0.1684,0.1809,0.0125
Recall,0.6851,0.6117,-0.0734


Ở ngưỡng 0,5: cảnh báo 16,713 hồ sơ, trong đó 3,024 vỡ nợ thật và 13,689 bị từ chối oan
Số khách nợ xấu bị bỏ sót: 1,920 / 4,944


**Nhận xét:**
- Nhìn cột "Chênh lệch" thì Random Forest **chưa vượt qua được** Logistic Regression. Quan trọng nhất là ROC-AUC (0,7548 so với 0,7616) và PR-AUC (0,2344 so với 0,2401) đều thấp hơn. Hai chỉ số này chấm khả năng **xếp đúng thứ tự rủi ro** của toàn bộ khách, không phụ thuộc vào ngưỡng nào cả, nên chúng cho biết mô hình nào thật sự phân biệt giỏi hơn.
- Vậy tại sao F1 và Precision của Random Forest lại cao hơn? Vì ở ngưỡng 0,5 nó **dè dặt hơn**: chỉ cảnh báo 16.713 hồ sơ, trong khi Logistic Regression cảnh báo khoảng 20.100. Cảnh báo ít thì tất nhiên tỷ lệ cảnh báo đúng cao hơn (hơn 18 trên 100 so với gần 17 trên 100). Đây là do chọn ngưỡng chứ không phải do mô hình khôn hơn.
- Cái giá của sự dè dặt đó nằm ở Recall 0,6117 so với 0,6851: Random Forest **bỏ sót 1.920 trong 4.944 khách nợ xấu**, nhiều hơn Logistic Regression 363 người. Đổi lại nó tránh oan cho khoảng 3.000 khách tốt (oan 13.689 so với khoảng 16.700). Với bài toán cho vay, một khách nợ xấu lọt lưới làm ngân hàng mất cả khoản vay, còn một khách tốt bị từ chối oan chỉ mất phần lãi đáng lẽ thu được — nên đánh đổi này đang đi sai hướng.
- **Đề xuất:** đừng kết luận mô hình nào tốt hơn dựa vào F1 hay Precision ở ngưỡng 0,5, vì con số đó thay đổi theo ngưỡng. Cứ giữ cả hai mô hình, so bằng ROC-AUC và PR-AUC ở Mục VII.1, rồi mới dò ngưỡng riêng cho mô hình được chọn ở Mục VIII.2.


### 3. Feature Importance

Đoạn code bên dưới xem Random Forest coi cột nào là quan trọng nhất, kèm số khách của từng nhóm như đã làm ở Mục IV.3.


In [24]:
# Moi cot duoc cham mot diem quan trong; tong diem cua tat ca cac cot bang 1
do_quan_trong = pd.Series(rf_model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)

# Lay 15 cot cao diem nhat, kem so khach de biet ket luan dua tren bao nhieu nguoi
top_15 = do_quan_trong.head(15)
bang_quan_trong = []
for ten_cot in top_15.index:
    bang_quan_trong.append({
        "Cột": ten_cot,
        "Độ quan trọng": round(float(top_15[ten_cot]), 4),
        "Phần trăm": f"{top_15[ten_cot] * 100:.2f}%",
        "Số khách thuộc nhóm (Train)": dem_khach_trong_nhom(ten_cot),
    })
display(pd.DataFrame(bang_quan_trong))

# Ve bieu do thanh ngang: cot cao diem nhat nam tren cung cho de doc
plt.figure(figsize=(9, 6))
plt.barh(top_15.index[::-1], top_15.values[::-1], color="#2E7D32")
plt.xlabel("Độ quan trọng (Feature Importance)")
plt.title("Top 15 cột quan trọng nhất theo Random Forest")
plt.tight_layout()
plt.show()

# Dem so cot gan nhu khong duoc dung den, de doi chieu voi so cot he so gan 0 cua Logistic Regression
so_cot_gan_0_rf = int((do_quan_trong < 0.0001).sum())

print(f"Tổng độ quan trọng của 15 cột đầu: {top_15.sum() * 100:.1f}% trên tổng 100%")
print(f"Số cột gần như không được dùng (độ quan trọng < 0,0001): {so_cot_gan_0_rf:,} / {len(do_quan_trong):,}")

# Tach rieng cot so va cot nhi phan trong top 15 de kiem tra xem co bi thien vi cot so khong
so_cot_so = sum(1 for ten in top_15.index if ten in num_cols)
print(f"Trong top 15: {so_cot_so} cột số và {15 - so_cot_so} cột nhị phân 0/1")


,Cột,Độ quan trọng,Phần trăm,Số khách thuộc nhóm (Train)
0,ext_ltv_interaction,0.1169,11.69%,(cột số)
1,ext_sources_mean,0.1148,11.48%,(cột số)
2,ext_sources_min,0.0745,7.45%,(cột số)
3,ext_source_2,0.0534,5.34%,(cột số)
4,ext_source_3,0.0525,5.25%,(cột số)
5,ext_sources_std,0.0263,2.63%,(cột số)
6,ext_source_1,0.0207,2.07%,(cột số)
7,bureau_avg_days_credit,0.0186,1.86%,(cột số)
8,employment_years,0.0180,1.80%,(cột số)
9,days_employed,0.0171,1.71%,(cột số)


Tổng độ quan trọng của 15 cột đầu: 58.5% trên tổng 100%
Số cột gần như không được dùng (độ quan trọng < 0,0001): 138 / 320
Trong top 15: 15 cột số và 0 cột nhị phân 0/1


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\2687654485.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- Top 15 gồm **15 cột số, 0 cột nhị phân**. Cách chấm điểm này đếm số lần một cột được dùng để cắt, mà cột số có hàng chục chỗ để cắt còn cột 0/1 chỉ có một — nên bảng thiên vị cột số.
- Bảy cột họ `ext_source` chiếm **45,9%**: gần một nửa quyết định dựa vào một nguồn thông tin. `ext_ltv_interaction` do nhóm tự tạo ở Mục III.3 đứng hạng 1 (11,69%).
- **138/320 cột** gần như không được dùng, so với 38 cột hệ số gần 0 của Logistic Regression.
- **Đề xuất:** bảng này chỉ cho biết mô hình hay cắt theo cột nào, chưa cho biết bỏ cột đó đi thì mô hình có tệ hơn không. Permutation Importance bên dưới trả lời câu đó.


Đoạn code bên dưới xáo trộn từng cột rồi đo xem điểm mô hình tụt bao nhiêu, để biết cột nào thật sự cần thiết.


In [25]:
# Chi do tren 20 cot cao diem nhat va 20.000 khach de chay trong vai phut
TOP_N = 20
SO_LAN_XAO = 5
top_20 = do_quan_trong.head(TOP_N).index

# Lay mau ngau nhien tu Test; y phai lay dung nhung khach do
mau_X_test = X_test_prep.sample(n=20_000, random_state=42)
mau_y_test = y_test.loc[mau_X_test.index]

# Diem PR-AUC khi chua dong gi, dung lam moc de so
diem_goc = average_precision_score(mau_y_test, rf_model.predict_proba(mau_X_test)[:, 1])

# Bo sinh so ngau nhien co hat giong co dinh de chay lai ra ket qua giong nhau
bo_ngau_nhien = np.random.default_rng(42)

bat_dau = time.time()
bang_xao_tron = []
for ten_cot in top_20:
    diem_tung_lan = []
    for _ in range(SO_LAN_XAO):
        # Xao tron rieng cot nay, cac cot khac giu nguyen
        ban_sao = mau_X_test.copy()
        ban_sao[ten_cot] = bo_ngau_nhien.permutation(ban_sao[ten_cot].to_numpy())
        # Cham diem lai sau khi cot nay da thanh vo nghia
        diem_tung_lan.append(average_precision_score(mau_y_test, rf_model.predict_proba(ban_sao)[:, 1]))
    # Diem tut cang nhieu nghia la cot do cang quan trong that
    bang_xao_tron.append({
        "Cột": ten_cot,
        "Hạng ở bảng trên": list(do_quan_trong.index).index(ten_cot) + 1,
        "Điểm tụt khi xáo trộn": round(diem_goc - np.mean(diem_tung_lan), 4),
        "Dao động giữa 5 lần": round(float(np.std(diem_tung_lan)), 4),
    })
thoi_gian_xao = time.time() - bat_dau

# Sap xep lai theo muc tut de thay thu tu moi khac thu tu cu the nao
bang_xao_tron = pd.DataFrame(bang_xao_tron).sort_values("Điểm tụt khi xáo trộn", ascending=False)
bang_xao_tron["Hạng mới"] = range(1, len(bang_xao_tron) + 1)
display(bang_xao_tron.reset_index(drop=True))

# Ve bieu do thanh ngang: cot lam diem tut nhieu nhat nam tren cung
plt.figure(figsize=(9, 7))
plt.barh(bang_xao_tron["Cột"][::-1], bang_xao_tron["Điểm tụt khi xáo trộn"][::-1], color="#00695C")
plt.xlabel("Mức điểm PR-AUC bị tụt khi xáo trộn cột")
plt.title("Top 20 cột theo Permutation Importance — Random Forest")
plt.tight_layout()
plt.show()

# Dem so cot xao tron xong ma diem gan nhu khong doi, tuc mo hinh khong thuc su can
so_cot_khong_anh_huong = int((bang_xao_tron["Điểm tụt khi xáo trộn"] < 0.001).sum())

print(f"PR-AUC gốc trên mẫu 20.000 khách: {diem_goc:.4f}")
print(f"Số cột trong top 20 mà xáo trộn xong điểm gần như không đổi: {so_cot_khong_anh_huong} / {TOP_N}")
print(f"Thời gian đo: {thoi_gian_xao / 60:.1f} phút")

,Cột,Hạng ở bảng trên,Điểm tụt khi xáo trộn,Dao động giữa 5 lần,Hạng mới
0,ext_ltv_interaction,1,0.0127,0.0014,1
1,ext_sources_mean,2,0.0118,0.0039,2
2,ext_source_3,5,0.0046,0.0013,3
3,ltv,11,0.0034,0.0012,4
4,ext_source_1,7,0.0028,0.0022,5
5,bureau_debt_ratio,12,0.0021,0.0005,6
6,ext_sources_min,3,0.0020,0.0031,7
7,ext_source_2,4,0.0019,0.0028,8
8,credit_term,13,0.0019,0.0012,9
9,days_employed,10,0.0014,0.0010,10


PR-AUC gốc trên mẫu 20.000 khách: 0.2271
Số cột trong top 20 mà xáo trộn xong điểm gần như không đổi: 7 / 20
Thời gian đo: 0.3 phút


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\4150423401.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- Hai cột đầu giữ nguyên hạng 1-2 ở cả hai cách tính, nên `ext_ltv_interaction` đúng là quan trọng thật chứ không phải chỉ hay được dùng để cắt. Feature tự tạo ở Mục III.3 có giá trị.
- Nhưng **7 trong 20 cột xáo trộn xong điểm gần như không đổi**, và `bureau_latest_days_credit` còn âm (−0,0002) — xáo lung tung mà mô hình lại nhỉnh lên chút ít. Lý do chính là **các cột trùng thông tin nhau**: `days_employed` và `employment_years` là cùng một dữ kiện tính theo ngày và theo năm, `age_years` và `days_birth` cũng vậy. Phá một cột thì cột kia gánh thay nên điểm không tụt.
- Xếp hạng đảo khá mạnh: `ltv` lên 11→4, `bureau_debt_ratio` 12→6, `credit_term` 13→9; ngược lại `ext_sources_min` tụt 3→7, `ext_source_2` 4→8, `employment_years` 9→12. Bảng impurity ở trên đã đánh giá cao quá mấy cột điểm ngoài và thời gian làm việc.
- Cột "Dao động giữa 5 lần" cho thấy phải đọc thận trọng: `ext_sources_mean` dao động 0,0039, lớn hơn cả điểm tụt của mọi cột từ hạng 3 trở xuống. Chỉ hai cột đầu là chắc chắn tách khỏi phần còn lại.
- **Đề xuất:** dùng bảng này chứ đừng dùng bảng impurity khi viết báo cáo, vì nó đo đúng cái ta quan tâm. Cột trùng thông tin nên loại bớt ở NB05 — giữ cả `days_employed` lẫn `employment_years` chỉ làm mô hình nặng thêm mà không thêm thông tin gì.


## VI. Mô hình HistGradientBoosting

### 1. Huấn luyện mô hình

Đoạn code bên dưới huấn luyện HistGradientBoosting, mô hình dựng cây nối tiếp nhau chứ không song song như Random Forest.


In [26]:
# Mo hinh nay cung cat du lieu theo nguong nen khong can doi don vi, dung ban goc
hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.05,      # moi cay chi duoc sua 5% loi con lai, sua tu tu cho chac
    max_iter=500,            # duoc dung toi da 500 cay, nhung co the dung som hon
    max_leaf_nodes=31,       # moi cay chi duoc dua ra toi da 31 ket luan
    min_samples_leaf=50,     # moi ket luan phai dua tren it nhat 50 khach, giong Random Forest
    early_stopping=True,     # tu biet khi nao nen dung lai
    validation_fraction=0.1, # cat 10% tap Train ra lam bai kiem tra rieng cho no
    n_iter_no_change=20,     # 20 cay lien tiep khong kha hon thi dung
    class_weight="balanced", # tinh nang khach no xau, giong hai mo hinh truoc
    random_state=42,
)

# Do thoi gian de doi chieu voi Logistic Regression va Random Forest
bat_dau = time.time()
hgb_model.fit(X_train_prep, y_train)
thoi_gian_hgb = time.time() - bat_dau

# So cay thuc te da dung; it hon 500 nghia la early stopping da bat dung lai
so_cay_da_dung = hgb_model.n_iter_

print(f"Thời gian huấn luyện: {thoi_gian_hgb:.1f} giây")
print(f"So sánh: Logistic Regression {thoi_gian_lr:.1f} giây, Random Forest {thoi_gian_rf:.1f} giây")
print(f"Số cây đã dùng: {so_cay_da_dung} / 500 cây cho phép")

# Diem tren 10% cat ra: dong dau tien la luc chi co 1 cay, dong cuoi la luc dung lai
diem_dau = hgb_model.validation_score_[0]
diem_cuoi = hgb_model.validation_score_[-1]
diem_tot_nhat = max(hgb_model.validation_score_)

print(f"\nĐiểm tự chấm trên 10% cắt ra — cây đầu tiên: {diem_dau:.4f}")
print(f"Điểm tự chấm khi dừng lại: {diem_cuoi:.4f} (tốt nhất từng đạt: {diem_tot_nhat:.4f})")

# Cham tran 500 cay nghia la no van con dang kha len thi bi cat ngang
assert so_cay_da_dung < 500, "Mô hình dùng hết 500 cây, cần tăng max_iter"


Thời gian huấn luyện: 27.7 giây
So sánh: Logistic Regression 6.8 giây, Random Forest 29.5 giây
Số cây đã dùng: 219 / 500 cây cho phép

Điểm tự chấm trên 10% cắt ra — cây đầu tiên: -0.6931
Điểm tự chấm khi dừng lại: -0.5664 (tốt nhất từng đạt: -0.5662)


**Nhận xét:**
- Điểm âm vì sklearn lấy **mức sai đổi dấu** cho thống nhất với các chỉ số khác: càng gần 0 càng tốt. Mốc −0,6931 của cây đầu tiên đúng bằng mức đoán bừa 50-50, đến khi dừng còn −0,5664, tức bớt sai 18%.
- Mô hình **dừng ở cây 219 trong 500 cây cho phép**: kỷ lục −0,5662 lập ở khoảng cây 199, rồi 20 cây sau không cây nào phá nổi nên `n_iter_no_change=20` bắt dừng. Early Stopping làm đúng việc và tiết kiệm hơn nửa số cây cho phép.
- Huấn luyện mất **ngang ngửa Random Forest dù chỉ dùng 219 cây so với 300**. Random Forest dựng cây độc lập nên chia được cho nhiều nhân CPU, còn ở đây mỗi cây phải chờ cây trước sửa lỗi xong — bù lại mỗi cây ở đây nhỏ hơn nhiều (31 lá so với 896 lá).
- **Lưu ý:** điểm −0,5664 chấm trên 10% cắt ra từ Train, chỉ để mô hình biết khi nào nên dừng. So sánh công bằng với hai mô hình kia nằm ở Mục VI.2.


### 2. Dự đoán và đánh giá trên tập Test

Đoạn code bên dưới chấm điểm HistGradientBoosting trên tập Test, rồi đặt cả ba mô hình cạnh nhau.

In [27]:
# Lay xac suat no xau ma mo hinh cham cho tung khach o tap Test
y_test_proba_hgb = hgb_model.predict_proba(X_test_prep)[:, 1]

# Cham diem tren Test de so sanh ba mo hinh; rieng nguong quyet dinh
# duoc do bang cross-validation trong tap Train o Muc VIII.2
chi_so_hgb_test = danh_gia("HistGradientBoosting", y_test, y_test_proba_hgb, mau="Oranges")

# Dat ca ba mo hinh canh nhau de thay ro hon kem o tung chi so
bang_ba_mo_hinh = pd.DataFrame({
    "Logistic Regression": chi_so_lr_test,
    "Random Forest": chi_so_rf_test,
    "HistGradientBoosting": chi_so_hgb_test,
}).round(4)

# Danh dau mo hinh cao diem nhat o tung dong de khoi phai do bang mat
bang_ba_mo_hinh["Cao nhất"] = bang_ba_mo_hinh.idxmax(axis=1)
display(bang_ba_mo_hinh)

# Doi xac suat thanh quyet dinh o nguong 0,5 de dem theo so ho so that
y_test_pred_hgb = (y_test_proba_hgb >= 0.5).astype(int)
so_bi_canh_bao = int(y_test_pred_hgb.sum())
so_canh_bao_dung = int(((y_test_pred_hgb == 1) & (y_test == 1)).sum())
so_canh_bao_oan = so_bi_canh_bao - so_canh_bao_dung
so_bo_sot = int(((y_test_pred_hgb == 0) & (y_test == 1)).sum())

print(f"Ở ngưỡng 0,5: cảnh báo {so_bi_canh_bao:,} hồ sơ")
print(f"  - Vỡ nợ thật: {so_canh_bao_dung:,}")
print(f"  - Bị từ chối oan: {so_canh_bao_oan:,}")
print(f"Số khách nợ xấu bị bỏ sót: {so_bo_sot:,} / {int(y_test.sum()):,}")

HistGradientBoosting | ROC-AUC: 0.7749 | PR-AUC: 0.2636

Báo cáo phân loại (ngưỡng 0.5000):
                 precision    recall  f1-score   support

Trả được nợ (0)       0.96      0.73      0.83     56093
     Nợ xấu (1)       0.18      0.69      0.29      4944

       accuracy                           0.72     61037
      macro avg       0.57      0.71      0.56     61037
   weighted avg       0.90      0.72      0.78     61037



C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\2189944949.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Logistic Regression,Random Forest,HistGradientBoosting,Cao nhất
ROC-AUC,0.7616,0.7548,0.7749,HistGradientBoosting
PR-AUC,0.2401,0.2344,0.2636,HistGradientBoosting
F1-Score,0.2704,0.2793,0.2863,HistGradientBoosting
Precision,0.1684,0.1809,0.1810,HistGradientBoosting
Recall,0.6851,0.6117,0.6851,Logistic Regression


Ở ngưỡng 0,5: cảnh báo 18,717 hồ sơ
  - Vỡ nợ thật: 3,387
  - Bị từ chối oan: 15,330
Số khách nợ xấu bị bỏ sót: 1,557 / 4,944


**Nhận xét:**
- HistGradientBoosting **thắng cả ROC-AUC (0,7749) lẫn PR-AUC (0,2636)** — hai chỉ số không phụ thuộc ngưỡng, tức nó xếp thứ tự rủi ro giỏi hơn thật chứ không phải ăn may ở ngưỡng 0,5. PR-AUC đáng chú ý nhất vì nợ xấu chỉ chiếm 8,1%: 0,2636 gấp 3,25 lần mốc đoán mò 0,0810, và hơn Logistic Regression 9,8%.
- Ở ngưỡng 0,5 nó bỏ sót đúng 1.557 trong 4.944 khách nợ xấu — **bằng đúng Logistic Regression**, vì cả hai tình cờ cùng Recall 0,6851. Nhưng nó chỉ cảnh báo oan 15.330 khách tốt, ít hơn Logistic Regression khoảng 1.400 người. Bắt được như nhau mà làm phiền ít hơn, đó là ưu thế thật.
- **Với bài toán cho vay thì bỏ sót là loại sai đắt nhất**: một khách nợ xấu lọt lưới làm mất cả khoản vay, còn một khách tốt bị từ chối oan chỉ mất phần lãi. PR-AUC cao hơn nghĩa là ở **mọi** mức bắt được nợ xấu, HistGradientBoosting đều cảnh báo oan ít hơn hai mô hình kia — hạ ngưỡng xuống thì nó bắt nhiều hơn mà vẫn oan ít hơn.
- **Đề xuất:** đây là ứng viên số 1 cho Mục VIII.1, và Mục VIII.2 phải dò ngưỡng theo hướng **ưu tiên bắt được nợ xấu**, chấp nhận cảnh báo oan nhiều hơn. Đổi lại, nó chậm hơn Logistic Regression khoảng 4 lần và khó giải thích nhất với người ngoài — Mục VII.3 sẽ cân nhắc cả ba mặt.


### 3. Feature Importance

Đoạn code bên dưới đo lại theo cách xáo trộn cột, trên đúng 20 cột và đúng mẫu khách đã dùng cho Random Forest ở Mục V.3.


In [28]:
# HistGradientBoosting khong co san bang do quan trong nhu Random Forest,
# nen o day chi do duoc bang cach xao tron tung cot

# Diem PR-AUC khi chua dong gi, dung lam moc de so
diem_goc_hgb = average_precision_score(mau_y_test, hgb_model.predict_proba(mau_X_test)[:, 1])

# Dung lai hat giong ngau nhien cu de ket qua lap lai duoc
bo_ngau_nhien = np.random.default_rng(42)

bang_hgb = []
for ten_cot in top_20:
    diem_tung_lan = []
    for _ in range(SO_LAN_XAO):
        # Xao tron rieng cot nay, cac cot khac giu nguyen
        ban_sao = mau_X_test.copy()
        ban_sao[ten_cot] = bo_ngau_nhien.permutation(ban_sao[ten_cot].to_numpy())
        # Cham diem lai sau khi cot nay da thanh vo nghia
        diem_tung_lan.append(average_precision_score(mau_y_test, hgb_model.predict_proba(ban_sao)[:, 1]))
    bang_hgb.append({
        "Cột": ten_cot,
        "Điểm tụt (HGB)": round(diem_goc_hgb - np.mean(diem_tung_lan), 4),
    })
bang_hgb = pd.DataFrame(bang_hgb)

# Ghep voi ket qua Random Forest o Muc V.3 de xem hai mo hinh co dua vao cung nhung cot khong
cot_rf = bang_xao_tron[["Cột", "Điểm tụt khi xáo trộn"]].rename(
    columns={"Điểm tụt khi xáo trộn": "Điểm tụt (RF)"}
)
bang_doi_chieu = bang_hgb.merge(cot_rf, on="Cột")
bang_doi_chieu = bang_doi_chieu.sort_values("Điểm tụt (HGB)", ascending=False)
display(bang_doi_chieu.reset_index(drop=True))

# Ve hai mo hinh canh nhau tren cung mot bieu do de nhin ra cho khac biet
vi_tri = np.arange(len(bang_doi_chieu))
plt.figure(figsize=(9, 8))
plt.barh(vi_tri + 0.2, bang_doi_chieu["Điểm tụt (HGB)"][::-1], height=0.4,
         color="#E65100", label="HistGradientBoosting")
plt.barh(vi_tri - 0.2, bang_doi_chieu["Điểm tụt (RF)"][::-1], height=0.4,
         color="#00695C", label="Random Forest")
plt.yticks(vi_tri, bang_doi_chieu["Cột"][::-1])
plt.xlabel("Mức điểm PR-AUC bị tụt khi xáo trộn cột")
plt.title("Cột nào thật sự quan trọng — so hai mô hình")
plt.legend()
plt.tight_layout()
plt.show()

# Dem so cot xao tron xong ma diem gan nhu khong doi, de so voi con so cua Random Forest
so_khong_doi_hgb = int((bang_doi_chieu["Điểm tụt (HGB)"] < 0.001).sum())

print(f"PR-AUC gốc của HistGradientBoosting trên mẫu 20.000 khách: {diem_goc_hgb:.4f}")
print(f"Số cột xáo trộn xong điểm gần như không đổi: {so_khong_doi_hgb} / {TOP_N}")

,Cột,Điểm tụt (HGB),Điểm tụt (RF)
0,ext_ltv_interaction,0.0359,0.0127
1,ext_sources_mean,0.0227,0.0118
2,credit_term,0.0101,0.0019
3,ext_source_1,0.0080,0.0028
4,age_years,0.0038,0.0009
5,ext_sources_min,0.0036,0.0020
6,bureau_debt_ratio,0.0032,0.0021
7,installments_payment_ratio,0.0027,0.0009
8,employment_years,0.0017,0.0010
9,days_birth,0.0016,0.0006


PR-AUC gốc của HistGradientBoosting trên mẫu 20.000 khách: 0.2595
Số cột xáo trộn xong điểm gần như không đổi: 6 / 20


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\2010235384.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- `ext_ltv_interaction` giữ hạng 1 ở cả hai mô hình, nhưng HistGradientBoosting tụt 0,0359 còn Random Forest chỉ tụt 0,0127 — **khai thác được gấp 2,8 lần**. Riêng cột này chiếm 13,8% điểm PR-AUC của HGB (0,2595 trên mẫu 20.000 khách). Feature nhóm tự tạo ở Mục III.3 được xác nhận lần thứ ba.
- Hai mô hình lấy cùng một thông tin qua hai đường khác nhau: Random Forest bám vào điểm gốc (`ext_source_3` tụt 0,0046), còn HistGradientBoosting bám vào bản tổng hợp (`ext_sources_mean` tụt 0,0227) và gần như bỏ qua `ext_source_2`, `ext_source_3` (chỉ 0,0012 và 0,0011). Không mô hình nào sai — chỉ là cách dùng khác nhau.
- HistGradientBoosting có **6/20 cột xáo trộn xong điểm không đổi, ít hơn con số 7/20 của Random Forest**, và mức tụt nói chung cũng lớn hơn (`credit_term` 0,0101 so với 0,0019). Nó vắt được nhiều thông tin hơn từ đúng bộ cột đó — đây chính là lý do nó thắng ở Mục VI.2.
- **Đề xuất:** 3 trong 6 cột đầu bảng (`ext_ltv_interaction`, `ext_sources_mean`, `ext_sources_min`) đều là feature nhóm tự tạo chứ không có sẵn trong dữ liệu gốc. Mục VII.4 đo hẳn đóng góp của chúng bằng cách bỏ đi rồi huấn luyện lại.


## VII. So sánh các mô hình

### 1. Bảng tổng hợp kết quả trên tập Test

Đoạn code bên dưới gom kết quả của ba mô hình thành hai bảng: bảng điểm số và bảng đặc điểm vận hành.


In [29]:
TEN_MO_HINH = ["Logistic Regression", "Random Forest", "HistGradientBoosting"]

# ==== Bang 1: diem so tren tap Test ====
bang_diem = pd.DataFrame(
    {"Logistic Regression": chi_so_lr_test, "Random Forest": chi_so_rf_test, "HistGradientBoosting": chi_so_hgb_test}
).round(4)

# Ghi thang ten mo hinh cao diem nhat de khoi phai do bang mat
bang_diem["Cao nhất"] = bang_diem[TEN_MO_HINH].idxmax(axis=1)

# Khoang cach giua mo hinh tot nhat va kem nhat o tung chi so
bang_diem["Chênh lệch cao - thấp"] = (
    bang_diem[TEN_MO_HINH].max(axis=1) - bang_diem[TEN_MO_HINH].min(axis=1)
).round(4)

print(f"Bảng 1 — Điểm số trên tập Test ({len(y_test):,} khách)")
display(bang_diem)

# ==== Bang 2: dac diem van hanh ====
# Dem so cot moi mo hinh thuc su dung den
cot_dung_lr = int((he_so.abs() >= 0.01).sum())
cot_dung_rf = int((do_quan_trong >= 0.0001).sum())

bang_van_hanh = pd.DataFrame(
    [
        {
            "Đặc điểm": "Thời gian huấn luyện (giây)",
            "Logistic Regression": f"{thoi_gian_lr:.1f}",
            "Random Forest": f"{thoi_gian_rf:.1f}",
            "HistGradientBoosting": f"{thoi_gian_hgb:.1f}",
        },
        {
            "Đặc điểm": f"Số cột thực sự dùng / {len(FEATURE_NAMES)}",
            "Logistic Regression": f"{cot_dung_lr:,}",
            "Random Forest": f"{cot_dung_rf:,}",
            "HistGradientBoosting": "Không đo trực tiếp được",
        },
        {
            "Đặc điểm": "Cần chuẩn hóa dữ liệu",
            "Logistic Regression": "Có",
            "Random Forest": "Không",
            "HistGradientBoosting": "Không",
        },
        {
            "Đặc điểm": "Giải thích cho người ngoài",
            "Logistic Regression": "Dễ (đọc thẳng hệ số)",
            "Random Forest": "Vừa (qua độ quan trọng)",
            "HistGradientBoosting": "Khó (phải xáo trộn cột)",
        },
    ]
).set_index("Đặc điểm")

print("\nBảng 2 — Đặc điểm vận hành")
display(bang_van_hanh)

# Dem xem moi mo hinh dan dau o bao nhieu chi so
so_lan_dan_dau = bang_diem["Cao nhất"].value_counts()
print("\nSố chỉ số mỗi mô hình dẫn đầu:")
for ten in TEN_MO_HINH:
    print(f"  {ten}: {so_lan_dan_dau.get(ten, 0)} / {len(bang_diem)}")

Bảng 1 — Điểm số trên tập Test (61,037 khách)


,Logistic Regression,Random Forest,HistGradientBoosting,Cao nhất,Chênh lệch cao - thấp
ROC-AUC,0.7616,0.7548,0.7749,HistGradientBoosting,0.0201
PR-AUC,0.2401,0.2344,0.2636,HistGradientBoosting,0.0292
F1-Score,0.2704,0.2793,0.2863,HistGradientBoosting,0.0159
Precision,0.1684,0.1809,0.1810,HistGradientBoosting,0.0126
Recall,0.6851,0.6117,0.6851,Logistic Regression,0.0734



Bảng 2 — Đặc điểm vận hành


,Logistic Regression,Random Forest,HistGradientBoosting
Đặc điểm,,,
Thời gian huấn luyện (giây),6.8,29.5,27.7
Số cột thực sự dùng / 320,282,182,Không đo trực tiếp được
Cần chuẩn hóa dữ liệu,Có,Không,Không
Giải thích cho người ngoài,Dễ (đọc thẳng hệ số),Vừa (qua độ quan trọng),Khó (phải xáo trộn cột)



Số chỉ số mỗi mô hình dẫn đầu:
  Logistic Regression: 1 / 5
  Random Forest: 0 / 5
  HistGradientBoosting: 4 / 5


**Nhận xét:**
- HistGradientBoosting dẫn đầu **4 trong 5 chỉ số**, và đó là các chỉ số đáng tin nhất: ROC-AUC, PR-AUC, F1-Score và Precision. Chỉ số duy nhất Logistic Regression không thua là Recall — nhưng thực ra hai mô hình bằng nhau đúng 0,6851, Random Forest mới là mô hình tụt lại (0,6117).
- Khoảng cách rất hẹp: **ROC-AUC giữa HistGradientBoosting và Logistic Regression chỉ chênh 0,0133**, PR-AUC chênh 0,0235. Ba mô hình phân biệt khách gần ngang nhau.
- Bảng 2 cho thấy cái giá phải trả: Logistic Regression huấn luyện **nhanh gấp khoảng 4 lần** (chưa tới 10 giây so với khoảng nửa phút) và là mô hình duy nhất đọc thẳng được hệ số để giải thích cho khách hàng. Random Forest chỉ dùng 182 trong 320 cột, còn Logistic Regression dùng tới 282 cột.
- **Đề xuất:** với chênh lệch chỉ 0,0133 ROC-AUC, đừng mặc định chọn mô hình điểm cao nhất. Nếu ngân hàng cần giải thích được lý do từ chối cho từng khách thì Logistic Regression vẫn là lựa chọn hợp lý. Mục VII.3 sẽ cân nhắc cả hai mặt trước khi chốt ở Mục VIII.1.


### 2. Đường cong ROC và Precision-Recall

Đoạn code bên dưới vẽ hai đường cong cho cả ba mô hình, rồi so điểm của chúng tại cùng một mức bắt được nợ xấu.


In [30]:
# Gom ba mo hinh vao mot danh sach de ve bang vong lap, moi mo hinh mot mau rieng
cac_mo_hinh = [
    ("Logistic Regression", y_test_proba_lr, "#1565C0"),
    ("Random Forest", y_test_proba_rf, "#2E7D32"),
    ("HistGradientBoosting", y_test_proba_hgb, "#E65100"),
]

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))

for ten, y_proba, mau in cac_mo_hinh:
    # Duong ROC: doi chieu ty le bat duoc no xau voi ty le canh bao oan
    ty_le_oan, ty_le_bat_duoc, _ = roc_curve(y_test, y_proba)
    ax_roc.plot(ty_le_oan, ty_le_bat_duoc, color=mau,
                label=f"{ten} (ROC-AUC {roc_auc_score(y_test, y_proba):.4f})")

    # Duong Precision-Recall: doi chieu ty le canh bao dung voi ty le bat duoc no xau
    do_chinh_xac, do_bao_phu, _ = precision_recall_curve(y_test, y_proba)
    ax_pr.plot(do_bao_phu, do_chinh_xac, color=mau,
               label=f"{ten} (PR-AUC {average_precision_score(y_test, y_proba):.4f})")

# Duong cheo la muc cua nguoi doan mo hoan toan
ax_roc.plot([0, 1], [0, 1], "--", color="gray", label="Đoán mò")
ax_roc.set_xlabel("Tỷ lệ khách tốt bị cảnh báo oan")
ax_roc.set_ylabel("Tỷ lệ khách nợ xấu bắt được")
ax_roc.set_title("Đường cong ROC — càng lên góc trên bên trái càng tốt")
ax_roc.legend(loc="lower right")

# Duong ngang la ty le no xau that, tuc muc cua nguoi canh bao bua moi ho so
ty_le_no_xau = y_test.mean()
ax_pr.axhline(ty_le_no_xau, ls="--", color="gray", label=f"Đoán mò ({ty_le_no_xau:.4f})")
ax_pr.set_xlabel("Tỷ lệ khách nợ xấu bắt được (Recall)")
ax_pr.set_ylabel("Tỷ lệ cảnh báo đúng (Precision)")
ax_pr.set_title("Đường cong Precision-Recall — càng lên cao càng tốt")
ax_pr.legend(loc="upper right")

plt.tight_layout()
plt.show()

# So ba mo hinh tai cung mot muc bat duoc no xau, de bo qua anh huong cua nguong
MUC_BAT_DUOC = 0.70
print(f"Khi cả ba cùng bắt được {MUC_BAT_DUOC:.0%} khách nợ xấu:")
for ten, y_proba, _ in cac_mo_hinh:
    do_chinh_xac, do_bao_phu, nguong = precision_recall_curve(y_test, y_proba)
    # Tim diem tren duong cong gan muc bat duoc 70% nhat
    vi_tri = int(np.argmin(np.abs(do_bao_phu - MUC_BAT_DUOC)))
    # Mang nguong ngan hon mang diem mot phan tu nen phai chan chi so
    nguong_can = nguong[min(vi_tri, len(nguong) - 1)]
    # Doi ty le canh bao dung thanh so ho so oan cho de hinh dung
    so_canh_bao = int(y_test.sum() * do_bao_phu[vi_tri] / do_chinh_xac[vi_tri])
    so_oan = so_canh_bao - int(y_test.sum() * do_bao_phu[vi_tri])
    print(f"  {ten}: ngưỡng {nguong_can:.4f} | cảnh báo đúng {do_chinh_xac[vi_tri]:.4f} | oan {so_oan:,} khách")

Khi cả ba cùng bắt được 70% khách nợ xấu:


  Logistic Regression: ngưỡng 0.4889 | cảnh báo đúng 0.1651 | oan 17,504 khách
  Random Forest: ngưỡng 0.4451 | cảnh báo đúng 0.1578 | oan 18,468 khách
  HistGradientBoosting: ngưỡng 0.4863 | cảnh báo đúng 0.1757 | oan 16,235 khách


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\70073972.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- Cách đọc đường Precision-Recall: chọn một mức trên trục ngang rồi dóng lên đường cong. Muốn bắt 70% khách nợ xấu thì tỷ lệ cảnh báo đúng chỉ còn khoảng 17,6%, tức **cứ 100 hồ sơ bị từ chối thì hơn 82 hồ sơ là oan**. Đường đi xuống từ trái sang phải chính là đánh đổi cốt lõi của bài toán: bắt càng nhiều nợ xấu thì càng oan nhiều khách tốt.
- Trên biểu đồ ROC ba đường gần như chồng lên nhau, đúng như khoảng cách hẹp giữa các mô hình ở Mục VII.1. Nhưng trên biểu đồ Precision-Recall thì **đường cam của HistGradientBoosting nằm trên hai đường kia gần như toàn tuyến** — nó tốt hơn ở mọi ngưỡng chứ không riêng ngưỡng 0,5.
- Ép cả ba cùng bắt 70% khách nợ xấu thì thấy rõ bằng số người: HistGradientBoosting cảnh báo oan **16.235 khách**, Logistic Regression 17.504, Random Forest 18.468. Chọn HistGradientBoosting là tránh oan được 1.269 khách tốt so với Logistic Regression, mà vẫn bắt được đúng bằng ấy khách nợ xấu. Điều này xác nhận nhận xét ở Mục VI.2.
- **Đề xuất:** báo cáo chỉ nên trích đường Precision-Recall. Đường ROC nhìn rất đẹp (phồng cao, AUC 0,77) nhưng che mất sự thật là tỷ lệ cảnh báo đúng không bao giờ vượt 30% — với bài toán nợ xấu 8,1% thì ROC luôn trông đẹp hơn thực tế. Cả ba mô hình đều cần ngưỡng khoảng 0,44-0,49 để đạt mức bắt 70% (HGB: 0,4863, LR: 0,4889, RF: 0,4451), nên Mục VIII.2 nên dò quanh vùng đó.


### 3. Nhận xét kết quả đánh giá

Đoạn code bên dưới chấm lại ba mô hình trên chính tập Train rồi đặt cạnh điểm Test, để xem mô hình có thuộc lòng dữ liệu đã học hay không.


In [31]:
# Cham lai ba mo hinh tren chinh tap Train de doi chieu voi diem Test
y_train_proba_lr = lr_model.predict_proba(X_train_scaled)[:, 1]
y_train_proba_rf = rf_model.predict_proba(X_train_prep)[:, 1]
y_train_proba_hgb = hgb_model.predict_proba(X_train_prep)[:, 1]

# Gom lai de duyet bang vong lap cho gon
xac_suat_train = {
    "Logistic Regression": y_train_proba_lr,
    "Random Forest": y_train_proba_rf,
    "HistGradientBoosting": y_train_proba_hgb,
}
chi_so_test = {
    "Logistic Regression": chi_so_lr_test,
    "Random Forest": chi_so_rf_test,
    "HistGradientBoosting": chi_so_hgb_test,
}

cac_dong = []
for ten, proba_train in xac_suat_train.items():
    # Diem tren tap da hoc thuoc
    roc_train = roc_auc_score(y_train, proba_train)
    pr_train = average_precision_score(y_train, proba_train)
    # Diem tren tap chua tung thay
    roc_test = chi_so_test[ten]["ROC-AUC"]
    pr_test = chi_so_test[ten]["PR-AUC"]
    cac_dong.append({
        "Mô hình": ten,
        "ROC-AUC Train": round(roc_train, 4),
        "ROC-AUC Test": round(roc_test, 4),
        "Khoảng cách ROC-AUC": round(roc_train - roc_test, 4),
        "PR-AUC Train": round(pr_train, 4),
        "PR-AUC Test": round(pr_test, 4),
        "Khoảng cách PR-AUC": round(pr_train - pr_test, 4),
    })

bang_hoc_vet = pd.DataFrame(cac_dong).set_index("Mô hình")

print(f"Tập Train: {len(y_train):,} khách — Tập Test: {len(y_test):,} khách")
display(bang_hoc_vet)

# Khoang cach cang lon nghia la mo hinh cang thuoc long tap Train
NGUONG_HOC_VET = 0.05
for ten in bang_hoc_vet.index:
    khoang_cach = bang_hoc_vet.loc[ten, "Khoảng cách ROC-AUC"]
    ket_luan = "học vẹt đáng lo" if khoang_cach > NGUONG_HOC_VET else "chấp nhận được"
    print(f"  {ten}: chênh {khoang_cach:+.4f} ROC-AUC — {ket_luan}")


Tập Train: 244,144 khách — Tập Test: 61,037 khách


,ROC-AUC Train,ROC-AUC Test,Khoảng cách ROC-AUC,PR-AUC Train,PR-AUC Test,Khoảng cách PR-AUC
Mô hình,,,,,,
Logistic Regression,0.7690,0.7616,0.0074,0.2499,0.2401,0.0097
Random Forest,0.8323,0.7548,0.0776,0.3441,0.2344,0.1097
HistGradientBoosting,0.8223,0.7749,0.0474,0.3162,0.2636,0.0526


  Logistic Regression: chênh +0.0074 ROC-AUC — chấp nhận được
  Random Forest: chênh +0.0776 ROC-AUC — học vẹt đáng lo
  HistGradientBoosting: chênh +0.0474 ROC-AUC — chấp nhận được


**Nhận xét:**
- Random Forest chênh **+0,0776 ROC-AUC** và **+0,1097 PR-AUC** giữa Train và Test — nó thuộc lòng tập Train nhiều hơn là rút ra được quy luật chung. Đây chính là lý do nó đứng cuối ở Mục VII.1 dù về bản chất mạnh hơn Logistic Regression.
- Logistic Regression gần như không học vẹt (**+0,0074**) vì nó chỉ vẽ được một đường phân chia đơn giản, không đủ chỗ để nhớ. HistGradientBoosting nằm giữa (**+0,0474**), vẫn trong mức chấp nhận được — công của cơ chế early stopping đã tự dừng ở Mục VI.1.
- **Đề xuất:** muốn cứu Random Forest thì phải siết `max_depth` xuống dưới 12 hoặc nâng `min_samples_leaf` lên trên 50. Nhưng nó đang thua cả hai mô hình kia trên tập Test nên không đáng bỏ thêm thời gian; loại khỏi danh sách ứng viên ở Mục VIII.1.


Đoạn code bên dưới quy kết quả của Mục VII.1 và VII.2 về bốn tiêu chí chọn mô hình, mỗi tiêu chí là một con số đã tính chứ không phải cảm nhận.


In [32]:
# Bon tieu chi de chon mo hinh, moi tieu chi do bang mot con so da tinh o tren
MUC_BAT_DUOC = 0.70

# Dem lai so khach tot bi canh bao oan khi ca ba cung bat duoc 70% no xau
so_oan_theo_mo_hinh = {}
for ten, y_proba in [("Logistic Regression", y_test_proba_lr),
                     ("Random Forest", y_test_proba_rf),
                     ("HistGradientBoosting", y_test_proba_hgb)]:
    do_chinh_xac, do_bao_phu, _ = precision_recall_curve(y_test, y_proba)
    # Diem tren duong cong gan muc bat duoc 70% nhat
    vi_tri = int(np.argmin(np.abs(do_bao_phu - MUC_BAT_DUOC)))
    so_bat_duoc = y_test.sum() * do_bao_phu[vi_tri]
    # Tong so ho so bi canh bao suy nguoc tu ty le canh bao dung
    so_canh_bao = so_bat_duoc / do_chinh_xac[vi_tri]
    so_oan_theo_mo_hinh[ten] = int(so_canh_bao - so_bat_duoc)

bang_ket_luan = pd.DataFrame({
    "PR-AUC trên Test (càng cao càng tốt)": {ten: chi_so_test[ten]["PR-AUC"] for ten in TEN_MO_HINH},
    "Khoảng cách Train - Test (càng thấp càng tốt)": bang_hoc_vet["Khoảng cách ROC-AUC"].to_dict(),
    "Số khách oan khi bắt 70% nợ xấu (càng thấp càng tốt)": so_oan_theo_mo_hinh,
    "Thời gian huấn luyện, giây (càng thấp càng tốt)": {
        "Logistic Regression": round(thoi_gian_lr, 1),
        "Random Forest": round(thoi_gian_rf, 1),
        "HistGradientBoosting": round(thoi_gian_hgb, 1),
    },
}).T

# Cot cuoi ghi thang ten mo hinh thang o tung tieu chi
bang_ket_luan["Mô hình tốt nhất"] = [
    bang_ket_luan.iloc[0][TEN_MO_HINH].idxmax(),   # PR-AUC: cao nhat thang
    bang_ket_luan.iloc[1][TEN_MO_HINH].idxmin(),   # ba tieu chi con lai: thap nhat thang
    bang_ket_luan.iloc[2][TEN_MO_HINH].idxmin(),
    bang_ket_luan.iloc[3][TEN_MO_HINH].idxmin(),
]

print("Bảng 3 — Bốn tiêu chí chọn mô hình")
display(bang_ket_luan)

# Bon tieu chi khong ngang gia tri nhau nen khong duoc dem phieu de chot.
# Hai tieu chi duoi day quy thang ra duoc so khach that, nen la tieu chi chinh.
TIEU_CHI_CHINH = [
    "PR-AUC trên Test (càng cao càng tốt)",
    "Số khách oan khi bắt 70% nợ xấu (càng thấp càng tốt)",
]

print("\nTiêu chí chính — quy ra được số khách thật:")
for ten_tieu_chi in TIEU_CHI_CHINH:
    print(f"  {ten_tieu_chi} → {bang_ket_luan.loc[ten_tieu_chi, 'Mô hình tốt nhất']}")

print("\nTiêu chí phụ — chi phí kỹ thuật, chỉ trả một lần khi huấn luyện:")
for ten_tieu_chi in bang_ket_luan.index:
    if ten_tieu_chi not in TIEU_CHI_CHINH:
        print(f"  {ten_tieu_chi} → {bang_ket_luan.loc[ten_tieu_chi, 'Mô hình tốt nhất']}")

# In them so phieu chi de thay ro rang dem phieu khong phan duoc thang thua
so_tieu_chi_thang = bang_ket_luan["Mô hình tốt nhất"].value_counts()
so_phieu = ", ".join(f"{ten} {so_tieu_chi_thang.get(ten, 0)}" for ten in TEN_MO_HINH)
print(f"\nNếu đếm phiếu ngang nhau thì kết quả là: {so_phieu} — không phân được thắng thua.")
print("Vì vậy quyết định dựa trên hai tiêu chí chính và được chốt ở Mục VIII.1.")


Bảng 3 — Bốn tiêu chí chọn mô hình


,Logistic Regression,Random Forest,HistGradientBoosting,Mô hình tốt nhất
PR-AUC trên Test (càng cao càng tốt),0.240142,0.234416,0.263639,HistGradientBoosting
Khoảng cách Train - Test (càng thấp càng tốt),0.007400,0.077600,0.047400,Logistic Regression
Số khách oan khi bắt 70% nợ xấu (càng thấp càng tốt),17504.000000,18468.000000,16235.000000,HistGradientBoosting
"Thời gian huấn luyện, giây (càng thấp càng tốt)",6.800000,29.500000,27.700000,Logistic Regression



Tiêu chí chính — quy ra được số khách thật:
  PR-AUC trên Test (càng cao càng tốt) → HistGradientBoosting
  Số khách oan khi bắt 70% nợ xấu (càng thấp càng tốt) → HistGradientBoosting

Tiêu chí phụ — chi phí kỹ thuật, chỉ trả một lần khi huấn luyện:
  Khoảng cách Train - Test (càng thấp càng tốt) → Logistic Regression
  Thời gian huấn luyện, giây (càng thấp càng tốt) → Logistic Regression

Nếu đếm phiếu ngang nhau thì kết quả là: Logistic Regression 2, Random Forest 0, HistGradientBoosting 2 — không phân được thắng thua.
Vì vậy quyết định dựa trên hai tiêu chí chính và được chốt ở Mục VIII.1.


**Nhận xét:**
- Kết quả **hòa 2-2**: HistGradientBoosting thắng ở PR-AUC (0,2636) và số khách oan (16.235), Logistic Regression thắng ở mức học vẹt (0,0074) và thời gian huấn luyện. Vì vậy không thể chốt mô hình bằng cách đếm phiếu — bốn tiêu chí này không ngang giá trị nhau.
- Cân theo mức độ quan trọng thì HistGradientBoosting hơn: hai tiêu chí nó thắng đều quy ra được người thật — ở cùng mức bắt 70% nợ xấu, nó làm phiền ít hơn Logistic Regression **1.269 khách** và ít hơn Random Forest **2.233 khách**. Hai tiêu chí Logistic Regression thắng chỉ là chi phí kỹ thuật: chậm hơn khoảng 4 lần, và chi phí đó chỉ trả đúng một lần khi huấn luyện.
- **Đề xuất:** chọn HistGradientBoosting, nhưng phải nêu rõ ở Mục VIII.1 rằng nó đổi lấy hai điểm yếu — học vẹt nhiều hơn Logistic Regression 6 lần và không đọc thẳng được lý do từ chối cho khách hàng.


### 4. Đánh giá đóng góp của các feature mới

Đoạn code bên dưới xếp 30 feature của NB05 vào 9 nhóm, tìm các cột chúng sinh ra trong ma trận đầu vào, rồi cộng điểm quan trọng mà Random Forest đã chấm ở Mục V.3.


In [33]:
# 30 feature NB05 tao ra, xep theo dung 9 nhom o bang ban giao cua NB05 (Muc VI.2)
NHOM_FEATURE_MOI = {
    "Tỷ số tài chính": ["ltv", "credit_term", "credit_to_income", "dti", "income_per_person"],
    "Đặc điểm khách hàng và hồ sơ": ["age_years", "employment_years", "employment_age_ratio",
                                     "id_publish_years", "application_hour_group"],
    "Điểm đánh giá ngoài": ["ext_sources_mean", "ext_sources_min", "ext_sources_std"],
    "Cờ lịch sử tín dụng": ["has_bureau", "has_previous", "has_installments",
                            "has_pos_cash", "has_credit_card"],
    "Lịch sử Bureau": ["bureau_debt_ratio", "has_bureau_overdue", "bureau_recency_days"],
    "Lịch sử Home Credit": ["previous_credit_to_current", "previous_recency_days"],
    "Lịch sử trả góp": ["installments_payment_ratio", "has_installments_late"],
    "POS/Cash và thẻ tín dụng": ["has_pos_cash_dpd", "credit_card_utilization"],
    "Interaction Features": ["age_income_interaction", "late_debt_interaction", "ext_ltv_interaction"],
}

# Gom 30 ten feature thanh mot danh sach phang
FEATURE_MOI = []
for cac_feature in NHOM_FEATURE_MOI.values():
    FEATURE_MOI.extend(cac_feature)


def tim_cot(ten_feature):
    """Tim cac cot trong ma tran dau vao sinh ra tu mot feature."""
    # Feature so giu nguyen ten nen tim thay ngay
    if ten_feature in FEATURE_NAMES:
        return [ten_feature]
    # Feature chu bi one-hot tach thanh nhieu cot mang tien to la ten goc
    return [cot for cot in FEATURE_NAMES if cot.startswith(ten_feature + "_")]


# Anh xa tung feature moi sang danh sach cot cua no
cot_cua_feature = {}
for ten_feature in FEATURE_MOI:
    cac_cot = tim_cot(ten_feature)
    assert cac_cot, f"Khong tim thay cot nao cua feature {ten_feature}"
    cot_cua_feature[ten_feature] = cac_cot

# Gom cot cua ca 30 feature moi, va phan con lai la cot goc
COT_FEATURE_MOI = []
for cac_cot in cot_cua_feature.values():
    COT_FEATURE_MOI.extend(cac_cot)
COT_GOC = [cot for cot in FEATURE_NAMES if cot not in set(COT_FEATURE_MOI)]

print(f"30 feature mới sinh ra {len(COT_FEATURE_MOI)} cột / {len(FEATURE_NAMES)} cột đầu vào")
print(f"Cột đến từ dữ liệu gốc: {len(COT_GOC)} cột")

# So sanh tong diem quan trong ma Random Forest cham cho hai ben
diem_moi = float(do_quan_trong[COT_FEATURE_MOI].sum())
diem_goc = float(do_quan_trong[COT_GOC].sum())

bang_ty_trong = pd.DataFrame([
    {"Nguồn cột": "30 feature mới của NB05", "Số cột": len(COT_FEATURE_MOI),
     "Tỷ trọng số cột (%)": round(len(COT_FEATURE_MOI) / len(FEATURE_NAMES) * 100, 1),
     "Tổng độ quan trọng (%)": round(diem_moi * 100, 1)},
    {"Nguồn cột": "Cột gốc từ dữ liệu thô", "Số cột": len(COT_GOC),
     "Tỷ trọng số cột (%)": round(len(COT_GOC) / len(FEATURE_NAMES) * 100, 1),
     "Tổng độ quan trọng (%)": round(diem_goc * 100, 1)},
]).set_index("Nguồn cột")

display(bang_ty_trong)

# Dem xem trong 20 cot manh nhat co bao nhieu cot den tu feature moi
top_20 = do_quan_trong.head(20)
so_moi_trong_top = sum(1 for cot in top_20.index if cot in set(COT_FEATURE_MOI))
print(f"\nTrong 20 cột mạnh nhất: {so_moi_trong_top} cột đến từ feature mới của NB05")
print("Các cột đó:", ", ".join(cot for cot in top_20.index if cot in set(COT_FEATURE_MOI)))


30 feature mới sinh ra 68 cột / 320 cột đầu vào
Cột đến từ dữ liệu gốc: 252 cột


,Số cột,Tỷ trọng số cột (%),Tổng độ quan trọng (%)
Nguồn cột,,,
30 feature mới của NB05,68,21.2,50.6
Cột gốc từ dữ liệu thô,252,78.8,49.4



Trong 20 cột mạnh nhất: 12 cột đến từ feature mới của NB05
Các cột đó: ext_ltv_interaction, ext_sources_mean, ext_sources_min, ext_sources_std, employment_years, ltv, bureau_debt_ratio, credit_term, installments_payment_ratio, age_years, bureau_recency_days, employment_age_ratio


**Nhận xét:**
- 30 feature mới chỉ chiếm **68/320 cột (21,2%)** nhưng gánh tới **50,6% tổng độ quan trọng** của Random Forest — nhiều hơn cả 252 cột gốc cộng lại. Trong 20 cột mạnh nhất có **12 cột là feature mới**, dẫn đầu là `ext_ltv_interaction` và bộ `ext_sources_mean/min/std`.
- Nhưng độ quan trọng chỉ cho biết mô hình **thích dùng** cột nào, không cho biết cột đó có **thêm thông tin mới** hay không. `ext_sources_mean` chỉ là trung bình của `ext_source_1/2/3`, mà cả ba cột gốc vẫn nằm trong 252 cột kia — điểm quan trọng của nó thực chất là điểm chia lại từ cột gốc.
- **Đề xuất:** không được dừng ở bảng này khi viết báo cáo. Phải kiểm bằng cách bỏ feature ra rồi huấn luyện lại như ở phần dưới, mới biết NB05 thật sự thêm được bao nhiêu.


Đoạn code bên dưới bỏ từng nhóm feature ra rồi huấn luyện lại để xem điểm tụt bao nhiêu. Tập đo là 20% cắt từ Train, nên tập Test vẫn còn nguyên cho Mục VIII.3.


In [34]:
# Cat 20% tap Train lam tap so sanh rieng cho muc nay; tap Test van danh cho Muc VIII.3
X_hoc, X_do, y_hoc, y_do = train_test_split(
    X_train_prep, y_train, test_size=0.2, random_state=42, stratify=y_train
)
print(f"Tập học tạm: {len(y_hoc):,} khách — Tập đo tạm: {len(y_do):,} khách")


def do_diem_khi_bo(cac_cot_bo):
    """Huan luyen lai HistGradientBoosting sau khi bo mot nhom cot, roi cham tren tap do tam."""
    # Giu lai nhung cot khong nam trong nhom bi bo
    cot_giu = [cot for cot in FEATURE_NAMES if cot not in set(cac_cot_bo)]
    # clone giu nguyen tham so nhung xoa sach nhung gi mo hinh goc da hoc
    mo_hinh = clone(hgb_model)
    mo_hinh.fit(X_hoc[cot_giu], y_hoc)
    xac_suat = mo_hinh.predict_proba(X_do[cot_giu])[:, 1]
    return roc_auc_score(y_do, xac_suat), average_precision_score(y_do, xac_suat)


bat_dau = time.time()

# Ban day du la moc de doi chieu moi lan bo bot
roc_day_du, pr_day_du = do_diem_khi_bo([])
print(f"Bản đầy đủ {len(FEATURE_NAMES)} cột: ROC-AUC {roc_day_du:.4f} | PR-AUC {pr_day_du:.4f}\n")

# Danh sach cac lan thu: bo tat ca feature moi truoc, roi bo lan luot tung nhom
cac_lan_thu = [("BỎ TẤT CẢ 30 feature mới", FEATURE_MOI)]
for ten_nhom, cac_feature in NHOM_FEATURE_MOI.items():
    cac_lan_thu.append((ten_nhom, cac_feature))

cac_dong = []
for ten_nhom, cac_feature in cac_lan_thu:
    # Gom toan bo cot sinh ra tu cac feature cua nhom nay
    cot_bo = []
    for ten_feature in cac_feature:
        cot_bo.extend(cot_cua_feature[ten_feature])
    roc, pr = do_diem_khi_bo(cot_bo)
    cac_dong.append({
        "Nhóm feature bị bỏ": ten_nhom,
        "Số feature": len(cac_feature),
        "Số cột bỏ": len(cot_bo),
        "ROC-AUC còn lại": round(roc, 4),
        "ROC-AUC mất đi": round(roc_day_du - roc, 4),
        "PR-AUC còn lại": round(pr, 4),
        "PR-AUC mất đi": round(pr_day_du - pr, 4),
    })
    print(f"  Xong: {ten_nhom}")

thoi_gian_ablation = time.time() - bat_dau

# Nhom nao bo di lam diem tut nhieu nhat thi xep tren
bang_ablation = (
    pd.DataFrame(cac_dong)
    .sort_values("ROC-AUC mất đi", ascending=False)
    .set_index("Nhóm feature bị bỏ")
)

print(f"\nTổng thời gian {len(cac_lan_thu) + 1} lần huấn luyện: {thoi_gian_ablation:.1f} giây")
print("Bảng 5 — Điểm mất đi khi bỏ từng nhóm feature (đo trên tập đo tạm cắt từ Train)")
display(bang_ablation)


Tập học tạm: 195,315 khách — Tập đo tạm: 48,829 khách


Bản đầy đủ 320 cột: ROC-AUC 0.7757 | PR-AUC 0.2639



  Xong: BỎ TẤT CẢ 30 feature mới


  Xong: Tỷ số tài chính


  Xong: Đặc điểm khách hàng và hồ sơ


...

  Xong: Lịch sử Home Credit


  Xong: Lịch sử trả góp


  Xong: POS/Cash và thẻ tín dụng


  Xong: Interaction Features

Tổng thời gian 11 lần huấn luyện: 237.0 giây
Bảng 5 — Điểm mất đi khi bỏ từng nhóm feature (đo trên tập đo tạm cắt từ Train)


,Số feature,Số cột bỏ,ROC-AUC còn lại,ROC-AUC mất đi,PR-AUC còn lại,PR-AUC mất đi
Nhóm feature bị bỏ,,,,,,
BỎ TẤT CẢ 30 feature mới,30,68,0.7726,0.0031,0.2623,0.0016
Tỷ số tài chính,5,5,0.7737,0.0019,0.2610,0.0029
Lịch sử Bureau,3,3,0.7754,0.0003,0.2623,0.0016
Cờ lịch sử tín dụng,5,5,0.7757,0.0000,0.2639,0.0000
Đặc điểm khách hàng và hồ sơ,5,7,0.7762,-0.0006,0.2644,-0.0006
Interaction Features,3,39,0.7763,-0.0006,0.2659,-0.0020
POS/Cash và thẻ tín dụng,2,2,0.7763,-0.0006,0.2646,-0.0008
Lịch sử trả góp,2,2,0.7768,-0.0011,0.2658,-0.0019
Lịch sử Home Credit,2,2,0.7770,-0.0013,0.2646,-0.0007


**Nhận xét:**
- Bỏ **cả 30 feature mới** chỉ làm ROC-AUC tụt từ 0,7757 xuống 0,7726, tức mất **0,0031** — chưa tới 0,4% giá trị. Đặt cạnh con số 50,6% độ quan trọng ở bảng trên thì khoảng cách rất lớn: mô hình **dùng** feature mới rất nhiều nhưng **không phụ thuộc** vào chúng.
- Nhóm duy nhất mất điểm thật là **Tỷ số tài chính**: -0,0019 ROC-AUC và -0,0029 PR-AUC. Đúng như dự đoán, vì `ltv`, `credit_term`, `credit_to_income`, `dti`, `income_per_person` đều là phép chia giữa hai cột — thứ mà mô hình cây rất khó tự dựng lại bằng các lát cắt lớn hơn / nhỏ hơn.
- Sáu nhóm còn lại bỏ ra thì điểm còn **nhỉnh lên**. Rõ nhất là **Điểm đánh giá ngoài**: bỏ ra được +0,0018 ROC-AUC dù cả 3 cột đều nằm trong top 20 quan trọng nhất — vì `ext_source_1/2/3` gốc vẫn còn nguyên, ba cột phái sinh chỉ là bản sao gây nhiễu. Interaction Features bỏ ra tiết kiệm được 39 cột mà điểm cũng không giảm.
- **Đề xuất:** chênh lệch dưới 0,002 nằm trong khoảng dao động của một lần chia tập, đừng tuyên bố "bỏ nhóm này thì mô hình tốt hơn". Kết luận an toàn: chỉ **Tỷ số tài chính** là chắc chắn đáng giữ, các nhóm còn lại giữ vì giúp diễn giải chứ không vì điểm số.


Đoạn code bên dưới xếp 9 nhóm theo mức điểm mất đi để thấy nhóm nào thực sự gánh mô hình.


In [35]:
# Bo dong "bo tat ca" ra khoi bieu do vi no la tong hop, khong phai mot nhom rieng
bang_ve = bang_ablation.drop(index="BỎ TẤT CẢ 30 feature mới").sort_values("ROC-AUC mất đi")

# Nhom nao mat diem thi to do, nhom nao bo di ma diem khong giam thi to xam
mau_cot = []
for gia_tri in bang_ve["ROC-AUC mất đi"]:
    mau_cot.append("#C62828" if gia_tri > 0 else "#9E9E9E")

plt.figure(figsize=(10, 6))
plt.barh(bang_ve.index, bang_ve["ROC-AUC mất đi"], color=mau_cot)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Số điểm ROC-AUC mất đi khi bỏ nhóm feature này")
plt.title("Đóng góp của từng nhóm feature mới — đo bằng cách bỏ ra rồi huấn luyện lại")
plt.tight_layout()
plt.show()

# Chia 9 nhom thanh hai loai de ket luan cho NB07
nhom_dang_giu = bang_ve[bang_ve["ROC-AUC mất đi"] > 0.0005].index.tolist()
nhom_khong_thiet = bang_ve[bang_ve["ROC-AUC mất đi"] <= 0.0005].index.tolist()

print("Nhóm bỏ ra là điểm tụt rõ (phải giữ và dựng lại ở NB07):")
for ten in reversed(nhom_dang_giu):
    print(f"  - {ten}: mất {bang_ve.loc[ten, 'ROC-AUC mất đi']:+.4f} ROC-AUC")

print("\nNhóm bỏ ra gần như không ảnh hưởng:")
for ten in nhom_khong_thiet:
    print(f"  - {ten}: mất {bang_ve.loc[ten, 'ROC-AUC mất đi']:+.4f} ROC-AUC")


Nhóm bỏ ra là điểm tụt rõ (phải giữ và dựng lại ở NB07):
  - Tỷ số tài chính: mất +0.0019 ROC-AUC

Nhóm bỏ ra gần như không ảnh hưởng:
  - Điểm đánh giá ngoài: mất -0.0018 ROC-AUC
  - Lịch sử Home Credit: mất -0.0013 ROC-AUC
  - Lịch sử trả góp: mất -0.0011 ROC-AUC
  - Đặc điểm khách hàng và hồ sơ: mất -0.0006 ROC-AUC
  - Interaction Features: mất -0.0006 ROC-AUC
  - POS/Cash và thẻ tín dụng: mất -0.0006 ROC-AUC
  - Cờ lịch sử tín dụng: mất +0.0000 ROC-AUC
  - Lịch sử Bureau: mất +0.0003 ROC-AUC


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\879464371.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- Biểu đồ chỉ có đúng **một cột đỏ**: bỏ Tỷ số tài chính thì mất điểm. Tám nhóm còn lại đều xám, nghĩa là bỏ ra không thiệt gì đo được.
- Cần nói rõ để tránh hiểu nhầm: mô hình cuối cùng **đã huấn luyện trên đủ 320 cột**, nên Notebook 07 vẫn phải dựng lại đủ 30 feature mới chấm được hồ sơ mới. Kết luận "có thể bỏ" ở đây chỉ dùng được nếu nhóm quyết định huấn luyện lại một bản gọn hơn.
- **Đề xuất:** giữ nguyên bộ 30 feature cho bản nộp, và đưa phát hiện này vào báo cáo như một kết luận của giai đoạn Feature Engineering: **giá trị của NB05 nằm ở chỗ biến dữ liệu thô thành các con số nghiệp vụ đọc được (tỷ lệ vay trên tài sản, tỷ lệ nợ trên thu nhập), chứ không nằm ở điểm số của mô hình.** Đây là câu trả lời trung thực hơn nhiều so với việc chỉ trưng bảng độ quan trọng 50,6%.


## VIII. Lựa chọn mô hình cuối cùng

### 1. Lý do lựa chọn mô hình

Đoạn code bên dưới chốt mô hình cuối cùng thành một biến dùng chung cho các mục sau, rồi đặt ba mô hình cạnh nhau lần cuối theo bốn bằng chứng đã đo ở Mục VII.


In [36]:
# Chot mo hinh cuoi cung. Tu day tro di cac muc sau chi goi bien nay,
# doi mo hinh khac thi chi phai sua dung mot cho.
TEN_MO_HINH_CUOI = "HistGradientBoosting"
MO_HINH_CUOI = hgb_model

# Mo hinh nay cat du lieu theo nguong lon hon / nho hon nen khong can doi don vi
CAN_CHUAN_HOA = False

# Chon dung ban du lieu ma mo hinh cuoi da hoc, tranh cham nham ban o cac muc sau
X_train_cuoi = X_train_scaled if CAN_CHUAN_HOA else X_train_prep
X_test_cuoi = X_test_scaled if CAN_CHUAN_HOA else X_test_prep

# Gom bon bang chung da do o Muc VII de doi chieu ba mo hinh mot lan cuoi
bang_ly_do = pd.DataFrame({
    "Điểm phân biệt (ROC-AUC Test)": {
        ten: round(chi_so_test[ten]["ROC-AUC"], 4) for ten in TEN_MO_HINH
    },
    "Điểm trên nhóm nợ xấu (PR-AUC Test)": {
        ten: round(chi_so_test[ten]["PR-AUC"], 4) for ten in TEN_MO_HINH
    },
    "Mức học vẹt (Train - Test)": bang_hoc_vet["Khoảng cách ROC-AUC"].to_dict(),
    "Số khách oan khi bắt 70% nợ xấu": so_oan_theo_mo_hinh,
    "Kết luận": {
        "Logistic Regression": "Loại — thua 0,0133 ROC-AUC, oan thêm 1.269 khách",
        "Random Forest": "Loại — học vẹt nặng nhất và điểm thấp nhất",
        "HistGradientBoosting": "CHỌN",
    },
}).T

print(f"Mô hình được chọn: {TEN_MO_HINH_CUOI}")
display(bang_ly_do)

# Kiem tra bien vua chot dung la mo hinh cao diem nhat o Muc VII.1, tranh chot nham tay
diem_cao_nhat = max(TEN_MO_HINH, key=lambda ten: chi_so_test[ten]["PR-AUC"])
assert diem_cao_nhat == TEN_MO_HINH_CUOI, f"Mô hình chốt không khớp mô hình cao điểm nhất: {diem_cao_nhat}"

print(f"\nBản dữ liệu dùng cho mô hình này: {'đã chuẩn hóa' if CAN_CHUAN_HOA else 'chưa chuẩn hóa (bản gốc)'}")
print(f"Kích thước: Train {X_train_cuoi.shape[0]:,} × {X_train_cuoi.shape[1]} cột")


Mô hình được chọn: HistGradientBoosting


,Logistic Regression,Random Forest,HistGradientBoosting
Điểm phân biệt (ROC-AUC Test),0.7616,0.7548,0.7749
Điểm trên nhóm nợ xấu (PR-AUC Test),0.2401,0.2344,0.2636
Mức học vẹt (Train - Test),0.0074,0.0776,0.0474
Số khách oan khi bắt 70% nợ xấu,17504,18468,16235
Kết luận,"Loại — thua 0,0133 ROC-AUC, oan thêm 1.269 khách",Loại — học vẹt nặng nhất và điểm thấp nhất,CHỌN



Bản dữ liệu dùng cho mô hình này: chưa chuẩn hóa (bản gốc)
Kích thước: Train 244,144 × 320 cột


**Nhận xét:**
- Chọn **HistGradientBoosting**. Nó dẫn đầu ở cả hai bằng chứng quy được ra người thật: PR-AUC 0,2636 (cao nhất) và số khách bị cảnh báo oan 16.235 (thấp nhất) khi cả ba cùng bắt 70% nợ xấu.
- **Loại Random Forest** trước tiên vì nó thua ở mọi mặt: điểm thấp nhất (ROC-AUC 0,7548) *và* học vẹt nặng nhất (chênh 0,0776 giữa Train và Test) *và* oan nhiều nhất (18.468 khách). Không có tiêu chí nào để bênh nó.
- **Loại Logistic Regression** khó hơn, vì nó ít học vẹt nhất (0,0074) và giải thích được từng hồ sơ. Nhưng chênh lệch 0,0133 ROC-AUC quy ra là **1.269 khách tốt bị từ chối oan thêm mỗi 61.037 hồ sơ** — con số này cụ thể hơn hẳn lợi thế "dễ giải thích".
- Từ đây trở đi các mục sau chỉ gọi `MO_HINH_CUOI` chứ không gọi thẳng `hgb_model`. Dòng `assert` đã kiểm tra biến vừa chốt đúng là mô hình PR-AUC cao nhất, nên không thể chốt nhầm tay mà notebook vẫn chạy qua.
- **Đề xuất:** ghi rõ trong báo cáo rằng đây là lựa chọn có điều kiện. Nếu ngân hàng yêu cầu giải thích được lý do từ chối cho từng khách hàng (điều luật một số nơi bắt buộc), thì phải quay lại Logistic Regression và chấp nhận mất 1.269 khách kia.


Đoạn code bên dưới ghi thẳng ba điểm yếu phải chấp nhận khi chọn mô hình này, kèm số đo và cách xử lý.


In [37]:
# Ba dieu phai chap nhan khi chon HistGradientBoosting, moi dieu kem con so da do
bang_danh_doi = pd.DataFrame([
    {
        "Điểm yếu": "Học vẹt nhiều hơn Logistic Regression",
        "Số đo": f"{bang_hoc_vet.loc[TEN_MO_HINH_CUOI, 'Khoảng cách ROC-AUC']:.4f} so với "
                 f"{bang_hoc_vet.loc['Logistic Regression', 'Khoảng cách ROC-AUC']:.4f}",
        "Cách xử lý": "Early stopping đã bật sẵn; Mục VIII.3 chấm lại trên tập Test để kiểm chứng",
    },
    {
        "Điểm yếu": "Không đọc thẳng được lý do từ chối cho từng khách",
        "Số đo": "Phải xáo trộn cột mới đo được, xem Mục VI.3",
        "Cách xử lý": "Kèm bảng Permutation Importance khi giải trình; nếu ngân hàng bắt buộc "
                      "giải thích từng hồ sơ thì phải quay lại Logistic Regression",
    },
    {
        "Điểm yếu": "Huấn luyện lâu hơn Logistic Regression",
        "Số đo": f"{thoi_gian_hgb / thoi_gian_lr:.1f} lần",
        "Cách xử lý": "Chi phí chỉ trả một lần khi huấn luyện, lúc chấm hồ sơ mới thì nhanh như nhau",
    },
]).set_index("Điểm yếu")

display(bang_danh_doi)

# Nhac lai muc chenh lech that su giua mo hinh duoc chon va mo hinh de giai thich nhat
chenh_roc = chi_so_test[TEN_MO_HINH_CUOI]["ROC-AUC"] - chi_so_test["Logistic Regression"]["ROC-AUC"]
chenh_oan = so_oan_theo_mo_hinh["Logistic Regression"] - so_oan_theo_mo_hinh[TEN_MO_HINH_CUOI]

print(f"Chênh lệch so với Logistic Regression: {chenh_roc:+.4f} ROC-AUC")
print(f"Quy ra người thật: tránh oan được {chenh_oan:,} khách tốt ở cùng mức bắt 70% nợ xấu")


,Số đo,Cách xử lý
Điểm yếu,,
Học vẹt nhiều hơn Logistic Regression,0.0474 so với 0.0074,Early stopping đã bật sẵn; Mục VIII.3 chấm lại...
Không đọc thẳng được lý do từ chối cho từng khách,"Phải xáo trộn cột mới đo được, xem Mục VI.3",Kèm bảng Permutation Importance khi giải trình...
Huấn luyện lâu hơn Logistic Regression,4.1 lần,"Chi phí chỉ trả một lần khi huấn luyện, lúc ch..."


Chênh lệch so với Logistic Regression: +0.0133 ROC-AUC
Quy ra người thật: tránh oan được 1,269 khách tốt ở cùng mức bắt 70% nợ xấu


**Nhận xét:**
- Điểm yếu nặng nhất là **học vẹt gấp 6,4 lần Logistic Regression** (0,0474 so với 0,0074). Con số này chưa tới mức báo động — vẫn dưới ngưỡng 0,05 đặt ở Mục VII.3 — nhưng nó nghĩa là điểm trên tập Train không dùng để báo cáo được, phải chờ Mục VIII.3 chấm trên tập Test.
- Điểm yếu về giải thích là điểm **không khắc phục được bằng kỹ thuật**, chỉ giảm nhẹ bằng cách kèm bảng Permutation Importance ở Mục VI.3. Đây là ràng buộc nghiệp vụ, không phải ràng buộc mô hình.
- Điểm yếu về tốc độ là nhẹ nhất: chậm hơn 3,8 lần nhưng chỉ trả một lần lúc huấn luyện. Khi chấm một hồ sơ mới ở NB07, cả ba mô hình đều trả lời trong vài phần nghìn giây.
- **Đề xuất:** đưa nguyên bảng này vào báo cáo thay vì chỉ khoe điểm số. Người chấm sẽ hỏi "vì sao không chọn mô hình dễ giải thích hơn" — bảng này trả lời sẵn, kèm điều kiện để đảo ngược quyết định.


### 2. Chọn ngưỡng dự đoán bằng cross-validation trên tập Train

Đoạn code bên dưới chia tập Train thành 5 phần để chấm điểm chéo, lấy xác suất out-of-fold làm cơ sở dò ngưỡng — nhờ vậy tập Test vẫn còn nguyên cho Mục VIII.3.


In [38]:
# Chia tap Train thanh 5 phan de moi khach deu co mot lan duoc cham boi mo hinh
# chua tung nhin thay minh; nho vay do duoc nguong ma khong dung den tap Test.
SO_PHAN = 5
chia_phan = StratifiedKFold(n_splits=SO_PHAN, shuffle=True, random_state=42)

# Cho san mot o trong cho moi khach o tap Train de dien xac suat vao
oof_proba = np.zeros(len(y_train))

# Doi chi so thanh mang de cat theo vi tri dong cho chac
X_train_arr = np.asarray(X_train_cuoi)
y_train_arr = np.asarray(y_train)

bat_dau = time.time()
for lan, (chi_so_hoc, chi_so_cham) in enumerate(chia_phan.split(X_train_arr, y_train_arr), start=1):
    # clone giu nguyen tham so nhung xoa sach nhung gi mo hinh goc da hoc
    mo_hinh_tam = clone(MO_HINH_CUOI)
    mo_hinh_tam.fit(X_train_arr[chi_so_hoc], y_train_arr[chi_so_hoc])
    # Chi cham diem cho phan bi giu lai, tuc phan mo hinh tam chua he hoc
    oof_proba[chi_so_cham] = mo_hinh_tam.predict_proba(X_train_arr[chi_so_cham])[:, 1]
    print(f"  Xong phan {lan}/{SO_PHAN}")

thoi_gian_cv = time.time() - bat_dau

# Diem tren xac suat out-of-fold: neu sat diem Test thi cach do nguong nay dang tin
roc_oof = roc_auc_score(y_train, oof_proba)
pr_oof = average_precision_score(y_train, oof_proba)

print(f"\nThời gian chạy {SO_PHAN} phần: {thoi_gian_cv:.1f} giây")
print(f"ROC-AUC out-of-fold: {roc_oof:.4f} (điểm trên tập Test ở Mục VI.2: {chi_so_test[TEN_MO_HINH_CUOI]['ROC-AUC']:.4f})")
print(f"PR-AUC out-of-fold:  {pr_oof:.4f} (điểm trên tập Test ở Mục VI.2: {chi_so_test[TEN_MO_HINH_CUOI]['PR-AUC']:.4f})")


  Xong phan 1/5


  Xong phan 2/5


  Xong phan 3/5


  Xong phan 4/5


  Xong phan 5/5

Thời gian chạy 5 phần: 105.8 giây
ROC-AUC out-of-fold: 0.7763 (điểm trên tập Test ở Mục VI.2: 0.7749)
PR-AUC out-of-fold:  0.2668 (điểm trên tập Test ở Mục VI.2: 0.2636)


**Nhận xét:**
- ROC-AUC out-of-fold **0,7763** và PR-AUC **0,2668**, chỉ lệch 0,0014 và 0,0032 so với điểm thật trên tập Test (0,7749 và 0,2636). Xác suất out-of-fold phản ánh gần đúng năng lực trên dữ liệu lạ, nên dò ngưỡng trên đó là đáng tin.
- Cái giá là 5 lần huấn luyện lại, tức khoảng 4 lần thời gian một lần huấn luyện đơn lẻ ở Mục VI.1.
- **Đề xuất:** giữ cách này thay vì cắt thêm một tập Validation riêng. Với 244.144 khách của tập Train, mỗi khách vẫn được chấm đúng một lần bởi mô hình chưa từng nhìn thấy mình, mà không phải hy sinh 20% dữ liệu huấn luyện.


Đoạn code bên dưới chọn ngưỡng theo Youden's J trên xác suất out-of-fold, rồi đặt cạnh vài mốc quen thuộc để thấy cái giá phải trả.


In [39]:
# Youden's J = ty le bat duoc no xau - ty le canh bao oan, chon diem J lon nhat
ty_le_oan_oof, ty_le_bat_duoc_oof, cac_nguong_oof = roc_curve(y_train, oof_proba)
chi_so_j = ty_le_bat_duoc_oof - ty_le_oan_oof
vi_tri_tot_nhat = int(np.argmax(chi_so_j))

# Day la nguong se mang sang tap Test o Muc VIII.3, khong duoc sua lai sau do
NGUONG_QUYET_DINH = float(cac_nguong_oof[vi_tri_tot_nhat])

print(f"Ngưỡng chốt theo Youden's J: {NGUONG_QUYET_DINH:.4f} (J = {chi_so_j[vi_tri_tot_nhat]:.4f})")
print(f"So với ngưỡng mặc định 0,5 thì {'thấp hơn' if NGUONG_QUYET_DINH < 0.5 else 'cao hơn'}"
      f" {abs(NGUONG_QUYET_DINH - 0.5):.4f}")


def do_hieu_qua(nguong):
    """Dem ket qua neu ap mot nguong len xac suat out-of-fold cua tap Train."""
    du_doan = (oof_proba >= nguong).astype(int)
    bat_dung = int(((du_doan == 1) & (y_train_arr == 1)).sum())
    canh_bao_oan = int(((du_doan == 1) & (y_train_arr == 0)).sum())
    bo_sot = int(((du_doan == 0) & (y_train_arr == 1)).sum())
    return {
        "Ngưỡng": round(nguong, 4),
        "Bắt được nợ xấu (%)": round(bat_dung / int(y_train_arr.sum()) * 100, 1),
        "Cảnh báo đúng (%)": round(bat_dung / max(bat_dung + canh_bao_oan, 1) * 100, 1),
        "Số khách bị oan": canh_bao_oan,
        "Số nợ xấu bỏ sót": bo_sot,
    }


# Dat nguong vua chon canh vai moc quen thuoc de thay ro cai gia phai tra
cac_nguong_thu = sorted({0.3, 0.4, round(NGUONG_QUYET_DINH, 4), 0.5, 0.6, 0.7})
bang_nguong = pd.DataFrame([do_hieu_qua(n) for n in cac_nguong_thu]).set_index("Ngưỡng")

print(f"\nBảng 4 — Hiệu quả từng ngưỡng trên {len(y_train):,} khách của tập Train (đo out-of-fold)")
display(bang_nguong)


Ngưỡng chốt theo Youden's J: 0.4730 (J = 0.4165)
So với ngưỡng mặc định 0,5 thì thấp hơn 0.0270



Bảng 4 — Hiệu quả từng ngưỡng trên 244,144 khách của tập Train (đo out-of-fold)


,Bắt được nợ xấu (%),Cảnh báo đúng (%),Số khách bị oan,Số nợ xấu bỏ sót
Ngưỡng,,,,
0.300,88.1,12.5,121781,2352
0.400,78.8,15.1,87490,4185
0.473,71.3,17.5,66448,5684
0.500,67.9,18.4,59714,6348
0.600,54.1,22.5,36857,9071
0.700,37.4,28.5,18501,12386


**Nhận xét:**
- Youden's J chốt ngưỡng **0,4730**, chỉ thấp hơn mức mặc định 0,5 đúng 0,027. Ngưỡng không phải hạ sâu vì mô hình đã dùng `class_weight="balanced"` ở Mục VI.1, xác suất vốn đã được kéo về giữa.
- Bảng 4 cho thấy cái giá của từng nấc: hạ từ 0,5 xuống 0,473 bắt thêm 3,4 điểm % nợ xấu (**664 khách nợ xấu**) nhưng phải cảnh báo oan thêm **6.734 khách tốt** — cứ chặn được 1 khoản vay xấu thì làm phiền khoảng 10 khách trả nợ tốt.
- **Đề xuất:** chốt 0,4730 làm ngưỡng chính thức và mang sang Mục VIII.3 chấm đúng một lần trên tập Test. Nếu nhóm muốn ưu tiên bắt nợ xấu mạnh hơn thì mốc 0,40 (bắt 78,8%) là lựa chọn kế tiếp, nhưng số khách oan nhảy từ 66.448 lên 87.490 — cần nhóm trưởng quyết chứ không tự đổi.


Đoạn code bên dưới quét toàn bộ dải ngưỡng để thấy hai đường đánh đổi chạy ngược chiều nhau ra sao.


In [40]:
# Quet nguong tu 0,05 den 0,95 de ve duong bat duoc no xau va duong canh bao dung
day_nguong = np.arange(0.05, 0.96, 0.01)
ket_qua_quet = pd.DataFrame([do_hieu_qua(n) for n in day_nguong])

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(ket_qua_quet["Ngưỡng"], ket_qua_quet["Bắt được nợ xấu (%)"],
        color="#C62828", label="Tỷ lệ bắt được nợ xấu (Recall)")
ax.plot(ket_qua_quet["Ngưỡng"], ket_qua_quet["Cảnh báo đúng (%)"],
        color="#1565C0", label="Tỷ lệ cảnh báo đúng (Precision)")

# Vach dung chi ro nguong da chot va nguong mac dinh de doi chieu
ax.axvline(NGUONG_QUYET_DINH, color="#2E7D32", ls="--",
           label=f"Ngưỡng chốt {NGUONG_QUYET_DINH:.4f}")
ax.axvline(0.5, color="gray", ls=":", label="Ngưỡng mặc định 0,5")

ax.set_xlabel("Ngưỡng quyết định (xác suất nợ xấu từ mức này trở lên thì từ chối)")
ax.set_ylabel("Tỷ lệ (%)")
ax.set_title("Đánh đổi giữa bắt được nợ xấu và cảnh báo đúng theo ngưỡng — đo out-of-fold trên tập Train")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Ghi lai de Muc VIII.3 va Muc VIII.4 dung dung mot con so nay
print(f"Ngưỡng mang sang Mục VIII.3 để chấm trên tập Test: {NGUONG_QUYET_DINH:.4f}")
print("Tập Test chưa bị dùng ở mục này — ngưỡng được dò hoàn toàn trên tập Train.")


Ngưỡng mang sang Mục VIII.3 để chấm trên tập Test: 0.4730
Tập Test chưa bị dùng ở mục này — ngưỡng được dò hoàn toàn trên tập Train.


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\1393339128.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- Hai đường chạy ngược chiều nhau đúng như dự đoán, nhưng đường xanh lên rất chậm: siết ngưỡng tới tận 0,7 thì tỷ lệ cảnh báo đúng cũng chỉ đạt 28,5%, tức vẫn hơn 7 trong 10 hồ sơ bị từ chối là oan. Đây là trần của bài toán chỉ có 8,1% nợ xấu, không phải lỗi của mô hình.
- Quanh vạch 0,4730 đường đỏ đang khá dốc — mỗi 0,01 ngưỡng đổi khoảng **1,3 điểm phần trăm** tỷ lệ bắt được nợ xấu. Vì vậy ngưỡng phải được ghi lại chính xác tới 4 chữ số, không được làm tròn thành 0,47 hay 0,5 khi bàn giao.
- **Đề xuất:** lưu đúng con số 0,4730 vào `model_metadata.json` ở Mục VIII.4 để Notebook 07 dùng lại, và đưa biểu đồ này kèm Bảng 4 vào báo cáo để giải thích vì sao chọn ngưỡng đó.


### 3. Đánh giá cuối cùng trên tập Test

Đoạn code bên dưới chấm mô hình đã chọn trên tập Test tại ngưỡng chốt ở Mục VIII.2. Đây là lần duy nhất tập Test được dùng để đánh giá kết quả cuối.


In [41]:
# Xac suat no xau ma mo hinh cuoi cham cho tung khach o tap Test
y_test_proba_cuoi = MO_HINH_CUOI.predict_proba(X_test_cuoi)[:, 1]

# Day phai la dung mo hinh da cham o Muc VI.2, khong duoc huan luyen lai giua chung
assert np.allclose(y_test_proba_cuoi, y_test_proba_hgb), "Mô hình cuối đã bị thay đổi so với Mục VI.2"

# Cham diem tai nguong da chot o Muc VIII.2, khong phai nguong mac dinh 0,5
chi_so_cuoi = danh_gia(
    f"{TEN_MO_HINH_CUOI} (ngưỡng chốt)",
    y_test, y_test_proba_cuoi,
    nguong=NGUONG_QUYET_DINH, mau="Oranges",
)

# Nguong duoc do tren Train, diem duoc cham tren Test — hai tap khac nhau nen phai doi chieu
print("Đối chiếu điểm đo trên Train (out-of-fold) và điểm thật trên Test:")
print(f"  ROC-AUC: {roc_oof:.4f} (Train) → {chi_so_cuoi['ROC-AUC']:.4f} (Test)")
print(f"  PR-AUC:  {pr_oof:.4f} (Train) → {chi_so_cuoi['PR-AUC']:.4f} (Test)")


HistGradientBoosting (ngưỡng chốt) | ROC-AUC: 0.7749 | PR-AUC: 0.2636

Báo cáo phân loại (ngưỡng 0.4730):
                 precision    recall  f1-score   support

Trả được nợ (0)       0.96      0.70      0.81     56093
     Nợ xấu (1)       0.17      0.71      0.28      4944

       accuracy                           0.70     61037
      macro avg       0.57      0.70      0.54     61037
   weighted avg       0.90      0.70      0.77     61037

Đối chiếu điểm đo trên Train (out-of-fold) và điểm thật trên Test:
  ROC-AUC: 0.7763 (Train) → 0.7749 (Test)
  PR-AUC:  0.2668 (Train) → 0.2636 (Test)


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_14584\2189944949.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:**
- Trên tập Test, mô hình đạt **ROC-AUC 0,7749** và **PR-AUC 0,2636** — chỉ thấp hơn mức đo out-of-fold trên tập Train đúng 0,0014 và 0,0032. Mô hình không tụt khi gặp dữ liệu chưa từng thấy, nên cách chọn mô hình ở Mục VIII.1 và cách dò ngưỡng ở Mục VIII.2 đều đứng vững.
- Ở ngưỡng 0,4730, Recall của nhóm nợ xấu là **0,71** còn Precision chỉ **0,17**: bắt được 71% khách nợ xấu, nhưng cứ 100 hồ sơ bị từ chối thì chỉ 17 hồ sơ vỡ nợ thật.
- Accuracy 0,70 **thấp hơn cả mức đoán bừa 91,9%** đã tính ở Mục II.3. Đây không phải lỗi mà là hệ quả cố ý: mô hình được chỉnh để bắt nợ xấu chứ không phải để đúng nhiều nhất. Đó cũng là lý do notebook không dùng Accuracy làm chỉ số đánh giá.
- **Đề xuất:** đây là bộ số chính thức để đưa vào báo cáo. Không được lấy lại điểm ở Mục VI.2 làm kết quả cuối, vì mục đó chấm ở ngưỡng mặc định 0,5 chứ không phải ngưỡng đã chốt.


Đoạn code bên dưới quy kết quả ra số khách thật và đặt cạnh ngưỡng mặc định 0,5 để thấy việc dò ngưỡng được gì.


In [42]:
def dem_ket_qua(nguong):
    """Quy mot nguong ra so khach that tren tap Test."""
    du_doan = (y_test_proba_cuoi >= nguong).astype(int)
    bat_dung = int(((du_doan == 1) & (y_test == 1)).sum())
    canh_bao_oan = int(((du_doan == 1) & (y_test == 0)).sum())
    bo_sot = int(((du_doan == 0) & (y_test == 1)).sum())
    return {
        "Số hồ sơ bị từ chối": bat_dung + canh_bao_oan,
        "Trong đó nợ xấu thật": bat_dung,
        "Trong đó bị từ chối oan": canh_bao_oan,
        "Số nợ xấu lọt lưới": bo_sot,
        "Tỷ lệ bắt được nợ xấu (%)": round(bat_dung / int(y_test.sum()) * 100, 1),
        "Tỷ lệ cảnh báo đúng (%)": round(bat_dung / max(bat_dung + canh_bao_oan, 1) * 100, 1),
    }


# Dat nguong chot canh nguong mac dinh de thay ro viec do nguong o Muc VIII.2 duoc gi
bang_cuoi = pd.DataFrame({
    f"Ngưỡng chốt ({NGUONG_QUYET_DINH:.4f})": dem_ket_qua(NGUONG_QUYET_DINH),
    "Ngưỡng mặc định (0,5000)": dem_ket_qua(0.5),
})
bang_cuoi["Chênh lệch"] = bang_cuoi.iloc[:, 0] - bang_cuoi.iloc[:, 1]

print(f"Kết quả cuối cùng trên {len(y_test):,} khách của tập Test "
      f"({int(y_test.sum()):,} khách nợ xấu thật)")
display(bang_cuoi)

# Nguong duoc do tren Train nen phai kiem xem sang Test co giu duoc muc bat duoc khong
ty_le_bat_duoc_test = bang_cuoi.loc["Tỷ lệ bắt được nợ xấu (%)"].iloc[0]
# Lay lai dung o tuong ung trong Bang 4 cua Muc VIII.2 thay vi chep tay con so
TY_LE_BAT_DUOC_TRAIN = bang_nguong.loc[round(NGUONG_QUYET_DINH, 4), "Bắt được nợ xấu (%)"]

print(f"\nMức bắt được nợ xấu: {TY_LE_BAT_DUOC_TRAIN}% khi dò trên Train "
      f"→ {ty_le_bat_duoc_test}% khi áp vào Test")
print(f"Lệch {abs(ty_le_bat_duoc_test - TY_LE_BAT_DUOC_TRAIN):.1f} điểm phần trăm")

# Gom cac con so cuoi cung de Muc VIII.4 ghi vao file ban giao
KET_QUA_CUOI = {
    "ten_mo_hinh": TEN_MO_HINH_CUOI,
    "nguong_quyet_dinh": round(float(NGUONG_QUYET_DINH), 4),
    "roc_auc_test": round(float(chi_so_cuoi["ROC-AUC"]), 4),
    "pr_auc_test": round(float(chi_so_cuoi["PR-AUC"]), 4),
    "recall_test": round(float(chi_so_cuoi["Recall"]), 4),
    "precision_test": round(float(chi_so_cuoi["Precision"]), 4),
    "f1_test": round(float(chi_so_cuoi["F1-Score"]), 4),
    "so_khach_test": int(len(y_test)),
}

print("\nBộ số liệu bàn giao cho Mục VIII.4:")
for khoa, gia_tri in KET_QUA_CUOI.items():
    print(f"  {khoa}: {gia_tri}")


Kết quả cuối cùng trên 61,037 khách của tập Test (4,944 khách nợ xấu thật)


,Ngưỡng chốt (0.4730),"Ngưỡng mặc định (0,5000)",Chênh lệch
Số hồ sơ bị từ chối,20625.0,18717.0,1908.0
Trong đó nợ xấu thật,3529.0,3387.0,142.0
Trong đó bị từ chối oan,17096.0,15330.0,1766.0
Số nợ xấu lọt lưới,1415.0,1557.0,-142.0
Tỷ lệ bắt được nợ xấu (%),71.4,68.5,2.9
Tỷ lệ cảnh báo đúng (%),17.1,18.1,-1.0



Mức bắt được nợ xấu: 71.3% khi dò trên Train → 71.4% khi áp vào Test
Lệch 0.1 điểm phần trăm

Bộ số liệu bàn giao cho Mục VIII.4:
  ten_mo_hinh: HistGradientBoosting
  nguong_quyet_dinh: 0.473
  roc_auc_test: 0.7749
  pr_auc_test: 0.2636
  recall_test: 0.7138
  precision_test: 0.1711
  f1_test: 0.276
  so_khach_test: 61037


**Nhận xét:**
- Ngưỡng chuyển từ Train sang Test gần như không xê dịch: mức bắt được nợ xấu **71,3% khi dò trên Train → 71,4% trên Test, lệch 0,1 điểm phần trăm**. Đây là bằng chứng cross-validation ở Mục VIII.2 đáng tin, và tập Test đúng là chưa hề bị dùng để dò ngưỡng.
- So với ngưỡng mặc định 0,5, ngưỡng chốt bắt thêm **142 khách nợ xấu** (số lọt lưới giảm từ 1.557 xuống 1.415) nhưng phải từ chối oan thêm **1.766 khách tốt**. Tức cứ chặn thêm được 1 khoản vay xấu thì làm phiền thêm khoảng 12 khách trả nợ tốt — sát với tỷ lệ khoảng 10 đo trên tập Train ở Mục VIII.2.
- Tỷ lệ cảnh báo đúng giảm từ 18,1% xuống 17,1%. Đây là cái giá đã biết trước của việc hạ ngưỡng, không phải dấu hiệu mô hình kém đi.
- **Đề xuất:** câu "hạ ngưỡng có đáng không" là quyết định **nghiệp vụ chứ không phải kỹ thuật**. Nếu thiệt hại từ 1 khoản nợ xấu lớn hơn 12 lần phần lãi mất đi từ 1 khách tốt bị từ chối thì hạ ngưỡng có lợi; ngược lại nên giữ 0,5. Nên đưa đúng con số 12 này cho nhóm quyết thay vì để notebook tự chốt.


### 4. Lưu mô hình và thông tin kèm theo

Đoạn code bên dưới lưu mô hình đã chọn và toàn bộ bộ tiền xử lý đi kèm vào thư mục `models/`.


In [43]:
# Bo tien xu ly gom moi thu NB07 can de dung lai dung ma tran dau vao tu ho so tho.
# Thieu bat ky manh nao trong day la NB07 khong the cham diem cho khach moi.
bo_tien_xu_ly = {
    # Bon buoc bien doi da hoc tu tap Train o Muc III.4 - III.6
    "num_imputer": num_imputer,
    "cat_imputer": cat_imputer,
    "encoder": encoder,
    "scaler": scaler,
    # Danh sach cot cua tung buoc, va thu tu cot cuoi cung
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "encoded_cols": list(encoded_cols),
    "feature_names": FEATURE_NAMES,
    "cot_chuan_hoa": cot_chuan_hoa,
    "cot_khoi_luong": COT_KHOI_LUONG,
    # Cac moc de dung lai hai bien tuong tac o Muc III.3
    "tuoi_bins": TUOI_BINS,
    "tuoi_labels": TUOI_LABELS,
    "nguong_thu_nhap": nguong_thu_nhap,
    "nhan_thu_nhap": NHAN_THU_NHAP,
    "nguong_du_no": nguong_du_no,
    "nhan_du_no": NHAN_DU_NO,
    "tre_han_cols": TRE_HAN_COLS,
    "nhan_thieu": NHAN_THIEU,
    # Mo hinh cuoi khong can doi don vi, ghi lai de NB07 khoi doan
    "can_chuan_hoa": CAN_CHUAN_HOA,
}

duong_dan_mo_hinh = MODELS_DIR / "model.pkl"
duong_dan_tien_xu_ly = MODELS_DIR / "preprocessor.pkl"

joblib.dump(MO_HINH_CUOI, duong_dan_mo_hinh)
joblib.dump(bo_tien_xu_ly, duong_dan_tien_xu_ly)

# Doi dung luong sang KB cho de doc
bang_file = pd.DataFrame([
    {"File": duong_dan_mo_hinh.name, "Nội dung": f"Mô hình {TEN_MO_HINH_CUOI} đã huấn luyện",
     "Dung lượng (KB)": round(duong_dan_mo_hinh.stat().st_size / 1024, 1)},
    {"File": duong_dan_tien_xu_ly.name, "Nội dung": f"{len(bo_tien_xu_ly)} thành phần tiền xử lý",
     "Dung lượng (KB)": round(duong_dan_tien_xu_ly.stat().st_size / 1024, 1)},
]).set_index("File")

print(f"Thư mục lưu: {MODELS_DIR}")
display(bang_file)


Thư mục lưu: D:\du an 1\models


,Nội dung,Dung lượng (KB)
File,,
model.pkl,Mô hình HistGradientBoosting đã huấn luyện,954.4
preprocessor.pkl,19 thành phần tiền xử lý,39.6


**Nhận xét:**
- `model.pkl` nặng 954,4 KB vì chứa cả 219 cây đã dựng. `preprocessor.pkl` chỉ 39,6 KB nhưng gói 19 thành phần — **nhẹ hơn mô hình 24 lần mà thiếu nó thì mô hình thành vô dụng**, vì không có cách nào dựng lại đúng 320 cột theo đúng thứ tự.
- Đường dẫn in ra là `D:\du an 1\models`, tức thư mục `models/` ở **gốc repo** chứ không phải `notebooks/models/`. Biến `PROJECT_ROOT` ở Mục II.1 dò theo vị trí `AGENTS.md` nên chạy từ VSCode, Jupyter hay `nbconvert` đều lưu về đúng một chỗ.
- **Đề xuất:** hai file `.pkl` này bị `.gitignore` chặn (`models/*`) nên **không lên GitHub**. Đồng đội muốn có thì phải chạy lại notebook này, hoặc nhóm thống nhất cho commit hai file (khoảng 1 MB) — việc này cần nhóm trưởng quyết.


Đoạn code bên dưới ghi tờ khai metadata đi kèm mô hình, lấy thẳng bộ số đã chốt ở Mục VIII.3.


In [44]:
# Metadata la to khai di kem mo hinh: ai doc cung biet mo hinh nay tu dau ra,
# cham diem the nao va phai dung nguong nao.
metadata = {
    "notebook_tao_ra": "06_modeling_evaluation.ipynb",
    "ngay_tao": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "mo_hinh": {
        "ten": TEN_MO_HINH_CUOI,
        "lop": type(MO_HINH_CUOI).__name__,
        "tham_so": {k: v for k, v in MO_HINH_CUOI.get_params().items() if v is not None},
        "so_cay_da_dung": int(MO_HINH_CUOI.n_iter_),
        "can_chuan_hoa_dau_vao": CAN_CHUAN_HOA,
    },
    "du_lieu": {
        "bang_nguon": "application_features",
        "so_dong_train": int(len(y_train)),
        "so_dong_test": int(len(y_test)),
        "ty_le_no_xau": round(float(y_test.mean()), 4),
        "so_cot_dau_vao": len(FEATURE_NAMES),
    },
    "cach_chon_nguong": (
        "Youden's J trên xác suất out-of-fold của cross-validation 5 phần trong tập Train "
        "(Mục VIII.2); tập Test chỉ dùng để chấm điểm cuối ở Mục VIII.3."
    ),
    "ket_qua_tren_test": KET_QUA_CUOI,
    "thu_tu_cot_dau_vao": FEATURE_NAMES,
}

duong_dan_metadata = MODELS_DIR / "model_metadata.json"
duong_dan_metadata.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Đã ghi {duong_dan_metadata.name} ({duong_dan_metadata.stat().st_size / 1024:.1f} KB)")
print("\nCác mục trong metadata:")
for khoa in metadata:
    print(f"  - {khoa}")

print(f"\nNgưỡng quyết định ghi trong file: {metadata['ket_qua_tren_test']['nguong_quyet_dinh']}")


Đã ghi model_metadata.json (12.5 KB)

Các mục trong metadata:
  - notebook_tao_ra
  - ngay_tao
  - mo_hinh
  - du_lieu
  - cach_chon_nguong
  - ket_qua_tren_test
  - thu_tu_cot_dau_vao

Ngưỡng quyết định ghi trong file: 0.473


**Nhận xét:**
- File metadata nặng 12,5 KB, phần lớn là danh sách `thu_tu_cot_dau_vao` gồm 320 tên cột. Nhìn thì thừa nhưng **bắt buộc phải có**: nếu NB07 dựng ma trận lệch thứ tự cột, mô hình vẫn chạy và vẫn trả ra xác suất, chỉ là sai — không có thông báo lỗi nào.
- Metadata ghi cả `cach_chon_nguong` chứ không chỉ con số 0,473. Sáu tháng sau đọc lại vẫn biết ngưỡng này dò bằng cross-validation trong tập Train, không phải dò trên tập Test.
- Đây là **file duy nhất trong `models/` được commit** lên GitHub, nên người đọc Pull Request vẫn nắm được mô hình ra sao dù không có hai file `.pkl` đi kèm.
- **Đề xuất:** NB07 khi nạp mô hình nên đối chiếu ma trận mình dựng với `thu_tu_cot_dau_vao` bằng một dòng `assert`, thay vì tin là đúng.


Đoạn code bên dưới đọc lại các file vừa lưu, dựng lại ma trận đầu vào từ đầu và chấm điểm để kiểm chứng.


In [45]:
# Luu xong chua chac dung. Doc lai tu dia va cham diem lai de kiem chung.
mo_hinh_doc_lai = joblib.load(duong_dan_mo_hinh)
tien_xu_ly_doc_lai = joblib.load(duong_dan_tien_xu_ly)

# Lay 1.000 khach dau tap Test lam bai kiem tra
SO_KHACH_KIEM = 1000
X_kiem = X_test.head(SO_KHACH_KIEM)

# Buoc 1: dien khuyet cot so bang trung vi da hoc
phan_so = pd.DataFrame(
    tien_xu_ly_doc_lai["num_imputer"].transform(X_kiem[tien_xu_ly_doc_lai["num_cols"]]),
    columns=tien_xu_ly_doc_lai["num_cols"], index=X_kiem.index,
)

# Buoc 2: dien khuyet cot chu, dua ve dang SimpleImputer doc duoc
cot_chu = X_kiem[tien_xu_ly_doc_lai["cat_cols"]]
cot_chu = cot_chu.astype(object).where(cot_chu.notna(), np.nan)
phan_chu = pd.DataFrame(
    tien_xu_ly_doc_lai["cat_imputer"].transform(cot_chu),
    columns=tien_xu_ly_doc_lai["cat_cols"], index=X_kiem.index,
)

# Buoc 3: ma hoa cot chu thanh cot 0/1 theo dung danh sach nhom da hoc
phan_ma_hoa = pd.DataFrame(
    tien_xu_ly_doc_lai["encoder"].transform(phan_chu),
    columns=tien_xu_ly_doc_lai["encoded_cols"], index=X_kiem.index,
)

# Buoc 4: ghep lai va xep dung thu tu cot ma mo hinh da hoc
ma_tran_kiem = pd.concat([phan_so, phan_ma_hoa], axis=1)[tien_xu_ly_doc_lai["feature_names"]]

# Buoc 5: doi don vi neu mo hinh yeu cau (mo hinh cuoi thi khong)
if tien_xu_ly_doc_lai["can_chuan_hoa"]:
    ma_tran_kiem[tien_xu_ly_doc_lai["cot_chuan_hoa"]] = tien_xu_ly_doc_lai["scaler"].transform(
        ma_tran_kiem[tien_xu_ly_doc_lai["cot_chuan_hoa"]]
    )

xac_suat_doc_lai = mo_hinh_doc_lai.predict_proba(ma_tran_kiem)[:, 1]

# Ket qua phai trung khit voi xac suat da tinh o Muc VIII.3
sai_lech_lon_nhat = float(np.abs(xac_suat_doc_lai - y_test_proba_cuoi[:SO_KHACH_KIEM]).max())
assert sai_lech_lon_nhat < 1e-9, f"Bộ file lưu ra cho kết quả khác: lệch {sai_lech_lon_nhat}"

print(f"Dựng lại từ file lưu và chấm điểm cho {SO_KHACH_KIEM:,} khách đầu tập Test")
print(f"Sai lệch lớn nhất so với Mục VIII.3: {sai_lech_lon_nhat:.2e} — coi như trùng khít")

# Doi chieu vai ho so cu the cho de kiem bang mat
bang_doi_chieu = pd.DataFrame({
    "Xác suất ở Mục VIII.3": y_test_proba_cuoi[:5].round(6),
    "Xác suất dựng lại từ file": xac_suat_doc_lai[:5].round(6),
    "Nhãn thật": y_test.head(5).values,
}, index=[f"Khách {ma}" for ma in X_kiem.index[:5]])

display(bang_doi_chieu)


Dựng lại từ file lưu và chấm điểm cho 1,000 khách đầu tập Test
Sai lệch lớn nhất so với Mục VIII.3: 0.00e+00 — coi như trùng khít


,Xác suất ở Mục VIII.3,Xác suất dựng lại từ file,Nhãn thật
Khách 141920,0.181103,0.181103,0
Khách 43039,0.766252,0.766252,0
Khách 280285,0.665909,0.665909,0
Khách 219640,0.456182,0.456182,0
Khách 158939,0.522266,0.522266,0


**Nhận xét:**
- Sai lệch lớn nhất là **0,00e+00** — trùng khít tuyệt đối chứ không phải xấp xỉ. Bộ file lưu ra đủ thành phần và đúng thứ tự cột, dựng lại từ đĩa cho ra đúng từng con số của Mục VIII.3.
- Đây là bài kiểm tra bắt lỗi thật chứ không phải thủ tục: thiếu một thành phần, lệch một cột hay quên bước đổi đơn vị đều làm dòng `assert` dừng notebook ngay tại đây.
- **Giới hạn cần nói rõ:** phép kiểm này bắt đầu từ `X_test` đã qua bước tạo biến tương tác (Mục III.3) và điền 0 theo cờ lịch sử (Mục III.4). NB07 nhận hồ sơ thô sẽ phải tự chạy hai bước đó bằng các mốc đã lưu trong `preprocessor.pkl` — **phần đó chưa được kiểm ở đây**.
- **Đề xuất:** NB07 nên lặp lại đúng phép so sánh này trên vài hồ sơ, nhưng đi từ dữ liệu thô, để phủ nốt hai bước còn thiếu.


### 5. Bàn giao sang Notebook 07

Đoạn code bên dưới in phiếu bàn giao cho Notebook 07, lấy thẳng từ các biến đã chốt và kiểm tra bộ tiền xử lý có đủ thành phần cho cả 5 bước hay không.


In [46]:
# Phieu ban giao duoc in tu bien that, khong chep tay, nen khong the lech voi ket qua o tren.
print("=== 1. BA FILE NB07 CẦN NẠP ===")
for duong_dan in [duong_dan_mo_hinh, duong_dan_tien_xu_ly, duong_dan_metadata]:
    print(f"  models/{duong_dan.name:<22} {duong_dan.stat().st_size / 1024:>8.1f} KB")

print("\n=== 2. NGƯỠNG QUYẾT ĐỊNH PHẢI DÙNG ===")
print(f"  {KET_QUA_CUOI['nguong_quyet_dinh']}  (xác suất từ mức này trở lên thì từ chối hồ sơ)")
print(f"  Dùng ngưỡng 0,5 sẽ bắt được ít nợ xấu hơn — xem Mục VIII.3")

print("\n=== 3. KẾT QUẢ MONG ĐỢI KHI CHẤM LẠI ===")
for khoa in ["roc_auc_test", "pr_auc_test", "recall_test", "precision_test"]:
    print(f"  {khoa}: {KET_QUA_CUOI[khoa]}")

print("\n=== 4. NĂM BƯỚC DỰNG LẠI MA TRẬN TỪ HỒ SƠ THÔ ===")
cac_buoc = [
    ("Tạo 2 biến tương tác", "tuoi_bins, nguong_thu_nhap, nguong_du_no, tre_han_cols", "Mục III.3"),
    ("Điền 0 theo cờ lịch sử", "cot_khoi_luong", "Mục III.4"),
    ("Điền khuyết cột số / cột chữ", "num_imputer, cat_imputer", "Mục III.4"),
    ("Mã hóa cột chữ thành 0/1", "encoder", "Mục III.5"),
    ("Xếp đúng thứ tự cột", "feature_names", "Mục III.5"),
]
for thu_tu, (viec, dung_gi, o_dau) in enumerate(cac_buoc, start=1):
    print(f"  Bước {thu_tu}. {viec}")
    print(f"          dùng: {dung_gi}   (cách làm xem {o_dau})")

# Mo hinh cuoi khong can doi don vi nen buoc scaler duoc bo qua, ghi ro de NB07 khoi lam thua
print(f"\n  Bước 6 (chỉ khi cần). Đổi đơn vị bằng scaler — mô hình hiện tại: "
      f"{'CÓ cần' if CAN_CHUAN_HOA else 'KHÔNG cần'}")

# Kiem tra bo tien xu ly du thanh phan cho ca 5 buoc tren, tranh ban giao thieu
thanh_phan_bat_buoc = [
    "tuoi_bins", "tuoi_labels", "nguong_thu_nhap", "nhan_thu_nhap",
    "nguong_du_no", "nhan_du_no", "tre_han_cols", "nhan_thieu",
    "cot_khoi_luong", "num_imputer", "cat_imputer", "encoder",
    "scaler", "num_cols", "cat_cols", "encoded_cols", "feature_names",
    "cot_chuan_hoa", "can_chuan_hoa",
]
thieu = [ten for ten in thanh_phan_bat_buoc if ten not in bo_tien_xu_ly]
assert not thieu, f"preprocessor.pkl thiếu thành phần: {thieu}"

print(f"\n=== 5. KIỂM TRA BỘ BÀN GIAO ===")
print(f"  preprocessor.pkl có đủ {len(thanh_phan_bat_buoc)}/{len(thanh_phan_bat_buoc)} thành phần cần thiết")
print(f"  Số cột mô hình chờ nhận: {len(FEATURE_NAMES)}")


=== 1. BA FILE NB07 CẦN NẠP ===
  models/model.pkl                 954.4 KB
  models/preprocessor.pkl           39.6 KB
  models/model_metadata.json        12.5 KB

=== 2. NGƯỠNG QUYẾT ĐỊNH PHẢI DÙNG ===
  0.473  (xác suất từ mức này trở lên thì từ chối hồ sơ)
  Dùng ngưỡng 0,5 sẽ bắt được ít nợ xấu hơn — xem Mục VIII.3

=== 3. KẾT QUẢ MONG ĐỢI KHI CHẤM LẠI ===
  roc_auc_test: 0.7749
  pr_auc_test: 0.2636
  recall_test: 0.7138
  precision_test: 0.1711

=== 4. NĂM BƯỚC DỰNG LẠI MA TRẬN TỪ HỒ SƠ THÔ ===
  Bước 1. Tạo 2 biến tương tác
          dùng: tuoi_bins, nguong_thu_nhap, nguong_du_no, tre_han_cols   (cách làm xem Mục III.3)
  Bước 2. Điền 0 theo cờ lịch sử
          dùng: cot_khoi_luong   (cách làm xem Mục III.4)
  Bước 3. Điền khuyết cột số / cột chữ
          dùng: num_imputer, cat_imputer   (cách làm xem Mục III.4)
  Bước 4. Mã hóa cột chữ thành 0/1
          dùng: encoder   (cách làm xem Mục III.5)
  Bước 5. Xếp đúng thứ tự cột
          dùng: feature_names   (cách làm xem Mục 

**Nhận xét:**
- Bộ bàn giao đủ **19/19 thành phần**, mô hình chờ nhận đúng **320 cột**. Dòng `assert` ở cuối cell kiểm tra danh sách này, nên nếu lần chạy sau có ai bỏ quên một thành phần trong `preprocessor.pkl` thì notebook dừng ngay tại đây chứ không để NB07 phát hiện.
- Hai bước đầu là chỗ dễ sai nhất: **Bước 1 (tạo biến tương tác)** và **Bước 2 (điền 0 theo cờ lịch sử)** không nằm trong phép kiểm chứng ở Mục VIII.4, vì phép kiểm đó bắt đầu từ dữ liệu đã qua hai bước này. NB07 phải tự dựng lại chúng bằng các mốc đã lưu, và nên tự kiểm bằng cách so xác suất với một vài hồ sơ trong tập Test.
- Mô hình cuối **không cần bước đổi đơn vị**, nên `scaler` tuy có trong file vẫn không được dùng. Ghi rõ điều này để NB07 khỏi chuẩn hóa thừa — chuẩn hóa nhầm thì mô hình vẫn chạy và vẫn ra xác suất, chỉ là sai.
- **Đề xuất:** dùng bốn chỉ số ở mục 3 làm bài kiểm tra nghiệm thu cho NB07. Nếu NB07 chấm lại tập Test mà không ra đúng ROC-AUC 0,7749 thì chắc chắn có bước dựng lại bị sai, không phải do dữ liệu.


**Cách NB07 nạp và dùng bộ bàn giao:**

```python
import joblib
from pathlib import Path

MODELS_DIR = Path("models")
mo_hinh = joblib.load(MODELS_DIR / "model.pkl")
tien_xu_ly = joblib.load(MODELS_DIR / "preprocessor.pkl")

# Dựng ma trận đầu vào theo đúng 5 bước ở trên, rồi:
ma_tran = ma_tran[tien_xu_ly["feature_names"]]        # xếp đúng thứ tự cột
xac_suat = mo_hinh.predict_proba(ma_tran)[:, 1]
tu_choi = xac_suat >= 0.4730                          # ngưỡng chốt ở Mục VIII.2
```

**Ba điều NB07 không được làm:**

| Không được | Vì sao |
|---|---|
| Huấn luyện lại mô hình | Kết quả ở Mục VIII.3 chỉ đúng với đúng mô hình trong `model.pkl` |
| Tự chọn ngưỡng khác 0,4730 | Ngưỡng này dò trên tập Train, đổi ngưỡng là mọi chỉ số đã báo cáo không còn đúng |
| `fit` lại imputer / encoder / scaler | Phải `transform` bằng bộ đã học từ Train, `fit` lại là rò rỉ dữ liệu |


## IX. Tổng kết

- NB06 đọc **305.181 dòng × 157 cột** từ bảng `application_features`, chia phân tầng thành Train **244.144** khách và Test **61.037** khách, tỷ lệ nợ xấu 8,1% giữ nguyên ở cả hai tập.
- Toàn bộ bước chuẩn bị đều **chỉ học từ tập Train** rồi áp sang Test: mốc chia nhóm của hai biến tương tác, trung vị điền khuyết, danh sách nhóm one-hot và mức chuẩn hóa. Sau mã hóa, ma trận đầu vào có **320 cột**.
- Huấn luyện và so sánh ba mô hình trên tập Test: **HistGradientBoosting ROC-AUC 0,7749 / PR-AUC 0,2636**, Logistic Regression 0,7616 / 0,2401, Random Forest 0,7548 / 0,2344. Khoảng cách giữa mô hình tốt nhất và Logistic Regression chỉ 0,0133 ROC-AUC.
- Đối chiếu điểm Train với điểm Test cho thấy **Random Forest học vẹt nặng nhất** (chênh 0,0776 ROC-AUC), HistGradientBoosting ở mức chấp nhận được (0,0474), Logistic Regression gần như không học vẹt (0,0074).
- Đo đóng góp của **30 feature do NB05 tạo ra** bằng cách bỏ ra rồi huấn luyện lại: chúng chiếm 50,6% độ quan trọng nhưng bỏ hết chỉ làm mất **0,0031 ROC-AUC**, vì phần lớn là biến phái sinh từ cột gốc vẫn còn trong dữ liệu. Nhóm duy nhất mất điểm thật là **Tỷ số tài chính** (`ltv`, `dti`, `credit_to_income`...).
- Chọn **HistGradientBoosting** dựa trên hai tiêu chí quy được ra người thật: PR-AUC cao nhất và số khách bị cảnh báo oan thấp nhất (16.235 so với 17.504 của Logistic Regression, ở cùng mức bắt 70% nợ xấu). Đổi lại phải chấp nhận học vẹt nhiều hơn và không giải thích được từng hồ sơ.
- Ngưỡng quyết định **0,4730** được dò bằng Youden's J trên xác suất out-of-fold của cross-validation 5 phần **trong tập Train**; tập Test chỉ dùng đúng một lần để chấm điểm cuối.
- Kết quả cuối trên tập Test tại ngưỡng đó: **Recall 0,7138 — Precision 0,1711 — F1 0,2760**. Mức bắt được nợ xấu dò trên Train là 71,3%, áp vào Test ra 71,4% — lệch 0,1 điểm phần trăm, cho thấy cách dò ngưỡng đáng tin.
- So với ngưỡng mặc định 0,5, ngưỡng chốt bắt thêm **142 khách nợ xấu** nhưng từ chối oan thêm **1.766 khách tốt** — cứ chặn thêm 1 khoản vay xấu thì làm phiền khoảng 12 khách trả nợ tốt. Đây là **quyết định nghiệp vụ**, notebook chỉ đưa ra con số chứ không chốt thay.
- Bàn giao ba file trong `models/`: `model.pkl`, `preprocessor.pkl` (19 thành phần đủ cho 5 bước dựng lại) và `model_metadata.json`. Đã kiểm chứng đọc lại từ đĩa và chấm điểm cho ra **đúng từng con số** của Mục VIII.3 (sai lệch 0,00e+00).
- **Hạn chế cần biết khi đọc kết quả:** ba mô hình được so sánh trên chính tập Test ở Mục VII, nên những chênh lệch nhỏ giữa chúng cần đọc dè dặt; Accuracy 0,70 thấp hơn mức đoán bừa 91,9% là hệ quả cố ý của việc ưu tiên bắt nợ xấu; và phép kiểm chứng bộ bàn giao chưa phủ hai bước đầu (tạo biến tương tác, điền 0 theo cờ lịch sử) nên NB07 phải tự kiểm.
- **Việc tiếp theo của NB07:** nạp ba file trên, dựng lại ma trận 320 cột từ hồ sơ thô theo đúng 5 bước ở Mục VIII.5, dùng ngưỡng 0,4730 và đối chiếu lại bốn chỉ số nghiệm thu trước khi đưa vào ứng dụng.
